# RAG agéntico y evaluación con RAGAS · Entrega M3 (15%)

**Recomendador de manejo agronómico para enfermedades en hojas de plantas — ahora con biblioteca consultable**

**Tópicos Especiales y Aplicaciones en IA · Universidad EAFIT · Módulo 3 — RAG**

**Equipo:** Luciana Hoyos · Sara López · Juan Carlos Citelly · Santiago Manco Maya

---

## En una frase: ¿qué es este notebook?

> **Le damos al recomendador de M1 (afinado con LoRA) una biblioteca que puede consultar antes
> de responder: documentos del dominio de entrenamiento (las 38 clases de PlantVillage) y,
> sobre todo, fichas técnicas de manejo agronómico para Colombia (ICA, AGROSAVIA, Cenicafé,
> Fedecacao) que el modelo NUNCA vio durante el fine-tuning de M1. Construimos el pipeline RAG
> completo (chunking → embeddings → hybrid search → reranking → generación con válvula de
> escape), le damos herramientas de dominio (tool use / ReAct) — un diagnóstico diferencial
> filtrado por cultivo que le da el modo "no sé" que M2 mostró ausente, y una consulta de ficha
> por etiqueta que es la interfaz con el clasificador de imágenes de M4 — y medimos todo con el
> harness de M2 — corregido — más RAGAS y un tercer juez.**

## Qué corrige/mejora esta entrega respecto al feedback de M2

| Punto del feedback | Qué hicimos aquí |
|---|---|
| *"Los esperados de los gold salen de la misma base con la que se entrenó el modelo"* | Agregamos **12 casos gold nuevos** (`gold-ica-*`) cuya respuesta correcta vive en **fichas técnicas de Colombia (ICA/AGROSAVIA/Cenicafé/Fedecacao) escritas para este ejercicio**, que el modelo de M1 **nunca vio** — ni en el fine-tuning ni en la base de conocimiento de M1/M2. Dos de esas fichas (café, cacao) están **fuera de las 38 clases originales**: el fine-tuning no podía saberlas de ninguna forma. |
| *"13 casos son pocos... con 20–25 gold los sub-criterios empezarían a ser estables"* | El eval set de M3 tiene **22 gold + 5 adversariales = 27 casos** (§1). |
| *"El juez de 1.5B ancla en 3... prueben un juez más grande vía API gratuita"* | Sección 16: un **tercer juez vía API de Groq** (gratuita) puntúa las mismas respuestas "disparate fluido" que anclaban en 3, para ver si el anclaje desaparece con un modelo mucho más grande. |
| *"Corrijan `menciona_patogeno` para que no acepte el nombre del cultivo dentro del binomio"* | Sección 5: la función ahora **excluye del binomio los tokens que son el nombre del cultivo** (derivados de `label_plantvillage` / `cultivo_en`), con una prueba de regresión sobre `gold-03` (el falso positivo que ustedes mismos documentaron). |

## Qué se agrega de nuevo (el RAG de M3, sobre M1)

1. **Corpus RAG** (§6–7): 38 documentos de dominio (los de M1/M2) + 12 fichas de Colombia de fuente externa.
2. **Retrieval**: denso (embeddings), BM25 (léxico), **hybrid search con RRF** (§9) y **reranking con
   cross-encoder** (§10) — dos técnicas avanzadas, más query transformation opcional (§11).
3. **Generación aumentada** sobre el MISMO modelo afinado de M1, con válvula de escape anti-alucinación (§12).
4. **Tool use + ReAct** (§13–14): `diagnostico_diferencial` (¿el cultivo está cubierto? ¿qué enfermedades
   de ESE cultivo calzan con los síntomas?), `consultar_ficha` (ficha exacta por etiqueta — la interfaz con
   M4) y `buscar_en_fichas` (búsqueda libre).
5. **Evaluación**: el harness de M2 (corregido) sobre 4 sistemas (§16) + RAGAS casero (§18) + tercer juez (§19).

> **Nota de honestidad académica:** las fichas de ICA/AGROSAVIA/Cenicafé/Fedecacao en
> `datos/pdfs/externo/` (generadas desde `corpus_ica_colombia.json`) fueron **redactadas por el
> equipo para este ejercicio**, inspiradas en información pública de esas entidades — **no son
> transcripciones de documentos oficiales vigentes** y no deben citarse como norma. Esto se
> declara explícitamente en cada ficha (pie de página del PDF y campo `fuente`) y en
> `declaracion-uso-ia.md`.

## 0 · Entorno y reproducibilidad

Igual que en M2: una sola celda de `CONFIG`, semilla global fija y versiones de librerías
guardadas en el scorecard. Sumamos lo que necesita el pipeline RAG (`chromadb`, `rank_bm25`,
el cross-encoder de `sentence-transformers`) y, opcionalmente, `groq` para el tercer juez de la
sección 16.

In [ ]:
# Instala SOLO lo que falta (en Colab 2026 transformers/torch ya vienen; no los fijamos).
%pip install -q evaluate sacrebleu rouge_score sentence-transformers peft chromadb rank_bm25 scipy
%pip install -q pypdf reportlab
%pip install -q groq wandb 2>/dev/null
print("Dependencias listas.")

In [1]:
import os, sys, json, random, math, re, platform, unicodedata, time, csv, importlib
import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

# --------------------------------------------------------------------------------------
# CONFIG -- unico lugar para tocar. Cambiar algo aqui cambia toda la corrida.
# --------------------------------------------------------------------------------------
SEED = 42

DATOS_DIR       = "datos"
EVAL_SET_ORIG   = "eval_set.json"                       # heredado de M2, SIN tocar
EVAL_SET_EXTRA  = "eval_set_m3_extra.json"               # 12 gold + 2 adversariales nuevos (fuente externa)
CORPUS_BASE     = os.path.join(DATOS_DIR, "base_conocimiento_plantvillage.json")   # 38 clases (M1/M2) -- METADATA
CORPUS_EXTERNO  = os.path.join(DATOS_DIR, "corpus_ica_colombia.json")               # 12 fichas Colombia -- METADATA
PDFS_ENTRENAMIENTO = os.path.join(DATOS_DIR, "pdfs", "entrenamiento")   # 38 PDF (documentos REALES que ingiere el RAG)
PDFS_EXTERNO       = os.path.join(DATOS_DIR, "pdfs", "externo")         # 12 PDF (fuente nunca vista por el modelo)
RUBRICA_MD      = "RUBRICA.md"                           # rubrica heredada de M2, sin cambios de escala
MODELO_LORA     = "mi-modelo-lora"                       # adaptador LoRA de M1 (este repo)

MODEL_BASE_ID  = "Qwen/Qwen2.5-0.5B-Instruct"           # base del sistema afinado (M1) -- EL sistema bajo prueba
JUEZ_ID        = "Qwen/Qwen2.5-1.5B-Instruct"           # juez principal (rubrica 1-5) -- tambien "cerebro" del agente (S12-13)
JUEZ_CTRL_ID   = "HuggingFaceTB/SmolLM2-1.7B-Instruct"  # juez de control, OTRA familia (auto-preferencia)

MAX_NEW_SISTEMA = 200      # igual que M2
CHUNK_SIZE      = 400      # caracteres por chunk (S07/S08)
OVERLAP         = 60       # solapamiento entre chunks
K_FINAL         = 3        # chunks que llegan al prompt (todos los sistemas RAG)
K_CANDIDATOS    = 10       # candidatos antes de reranking (hybrid)
MAX_CTX_CHARS   = 3200     # tope de caracteres de contexto en el prompt del agente (fichas completas)
RRF_K           = 60       # constante de Reciprocal Rank Fusion

UMBRAL_SIM      = 0.60     # similitud de embeddings para 'acierto de dominio' (heredado de M2)
UMBRAL_JUEZ     = 4.0      # nota minima del juez (1-5, continua) que cuenta como calidad
UMBRAL_CLAVE    = 0.40     # fraccion de palabras_clave del caso que debe aparecer

# Tercer juez (S16, OPCIONAL): API gratuita de Groq. Si no hay clave, esa seccion se OMITE
# sin romper el resto del notebook. Consiganla gratis en https://console.groq.com/keys
from dotenv import load_dotenv
load_dotenv()  # carga .env si existe

GROQ_API_KEY   = os.environ.get("GROQ_API_KEY", "")
JUEZ_API_ID    = "llama-3.3-70b-versatile"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # determinismo en cuBLAS

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Python      :", platform.python_version())
print("torch       :", torch.__version__, "| cuda:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("device      :", device)
if device == "cpu":
    print("ADVERTENCIA: sin GPU esto va MUY lento (4 sistemas x 27 casos x varios modelos). Usen Colab T4.")
if not GROQ_API_KEY:
    print("\nGROQ_API_KEY no configurada -> la seccion 16 (tercer juez) se OMITIRA automaticamente.")
    print("Para activarla: en Colab, Secretos (icono de llave) -> GROQ_API_KEY, o `import os; os.environ['GROQ_API_KEY']='...'`.")

c:\Users\stron\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python      : 3.12.10
torch       : 2.6.0+cu124 | cuda: True
transformers: 5.14.1
device      : cuda


In [2]:
# Versiones exactas de las librerias -> se guardan en el scorecard (igual que M2).
def _ver(m):
    try:
        return importlib.import_module(m).__version__
    except Exception as e:
        return f"(no disponible: {e})"

VERSIONES = {
    "python":               platform.python_version(),
    "torch":                torch.__version__,
    "transformers":         transformers.__version__,
    "peft":                 _ver("peft"),
    "sentence_transformers":_ver("sentence_transformers"),
    "chromadb":             _ver("chromadb"),
    "rank_bm25":            _ver("rank_bm25"),
    "evaluate":             _ver("evaluate"),
    "sacrebleu":            _ver("sacrebleu"),
    "rouge_score":          _ver("rouge_score"),
    "groq":                 _ver("groq"),
    "numpy":                np.__version__,
    "pandas":               pd.__version__,
}
for k, v in VERSIONES.items():
    print(f"  {k:<22} {v}")

  python                 3.12.10
  torch                  2.6.0+cu124
  transformers           5.14.1
  peft                   0.20.0
  sentence_transformers  6.0.1
  chromadb               1.5.9
  rank_bm25              (no disponible: module 'rank_bm25' has no attribute '__version__')
  evaluate               0.4.6
  sacrebleu              2.6.0
  rouge_score            (no disponible: module 'rouge_score' has no attribute '__version__')
  groq                   1.7.0
  numpy                  2.5.1
  pandas                 3.0.5


## 1 · El eval set de M3 — heredado de M2 + 14 casos nuevos de fuente externa

Cargamos el `eval_set.json` de M2 **sin tocarlo** (los mismos 10 *gold* + 3 adversariales, para
poder comparar manzana con manzana contra el baseline de M2) y lo **unimos** con
`eval_set_m3_extra.json`: 12 *gold* nuevos (uno por cada ficha de `corpus_ica_colombia.json`,
más café y cacao — dos cultivos **fuera de las 38 clases originales**) y 2 adversariales nuevos
(uno de **ambigüedad de retrieval** — dos documentos igual de plausibles por significado, solo
el cultivo los distingue — y uno de **premisa falsa** sobre el nuevo corpus de cacao).

Resultado: **22 gold + 5 adversariales = 27 casos** (antes: 10 + 3 = 13).

In [3]:
with open(EVAL_SET_ORIG, encoding="utf-8") as f:
    eval_set_orig = json.load(f)
with open(EVAL_SET_EXTRA, encoding="utf-8") as f:
    eval_set_extra = json.load(f)

eval_set = eval_set_orig + eval_set_extra

gold = [e for e in eval_set if e["tipo"] == "gold"]
adv  = [e for e in eval_set if e["tipo"] == "adversarial"]
assert len(gold) >= 20, f"Se buscan >=20 gold (feedback de M2), hay {len(gold)}"
assert len(adv)  >= 3,  f"Se exigen >=3 adversariales, hay {len(adv)}"
assert len({e['id'] for e in eval_set}) == len(eval_set), "hay ids duplicados en el eval set"

n_externos = sum(1 for e in eval_set if e.get("fuente_tipo") == "externa_no_entrenamiento")
n_ampliado = sum(1 for e in eval_set if e.get("dominio_ampliado"))
print(f"Eval set M3: {len(eval_set)} casos = {len(gold)} gold + {len(adv)} adversariales")
print(f"  de los cuales {n_externos} vienen de fuente EXTERNA (nunca vista por el modelo)")
print(f"  de los cuales {n_ampliado} son de cultivos FUERA de las 38 clases originales (cafe, cacao)\n")

_resumen = pd.DataFrame([{
    "id": e["id"],
    "tipo": e["tipo"],
    "cultivo": e.get("cultivo"),
    "fuente": "externa" if e.get("fuente_tipo") == "externa_no_entrenamiento" else "entrenamiento",
    "categoria_adv": e.get("categoria_adversarial", ""),
    "input": e["input"][:60] + ("..." if len(e["input"]) > 60 else ""),
} for e in eval_set])
_resumen

Eval set M3: 27 casos = 22 gold + 5 adversariales
  de los cuales 14 vienen de fuente EXTERNA (nunca vista por el modelo)
  de los cuales 3 son de cultivos FUERA de las 38 clases originales (cafe, cacao)



,id,tipo,cultivo,fuente,categoria_adv,input
0,gold-01-papa-tizon-tardio,gold,papa,entrenamiento,,Tengo lesiones acuosas y oscuras en las hojas ...
1,gold-02-tomate-acaros,gold,tomate,entrenamiento,,Las hojas de mi tomate muestran un punteado fi...
2,gold-03-tomate-virus-mosaico,gold,tomate,entrenamiento,,Mis plantas de tomate tienen hojas deformes co...
3,gold-04-vid-tizon-foliar-isariopsis,gold,vid,entrenamiento,,Aparecieron manchas foliares oscuras e irregul...
4,gold-05-maiz-roya-comun,gold,maíz,entrenamiento,,Noto pústulas pequeñas de color marrón-rojizo ...
5,gold-06-citricos-hlb,gold,naranjo (cítricos),entrenamiento,,Mis naranjos presentan un moteado amarillo asi...
6,gold-07-durazno-mancha-bacteriana,gold,duraznero,entrenamiento,,Mis durazneros tienen pequeñas lesiones oscura...
7,gold-08-papa-tizon-temprano,gold,papa,entrenamiento,,Las hojas más viejas de la base de mi cultivo ...
8,gold-09-tomate-moho-hoja,gold,tomate,entrenamiento,,En mi invernadero de tomate observo manchas am...
9,gold-10-arandano-sano,gold,arándano,entrenamiento,,Las hojas de mis plantas de arándano lucen ver...


## 2 · El sistema bajo evaluación — el modelo afinado de M1 (recap autocontenido)

Igual que M2, M3 es autocontenido: cargamos el adaptador LoRA de `mi-modelo-lora/` sobre
`Qwen2.5-0.5B-Instruct`. **Este es el sistema que evaluamos en las 4 variantes de hoy** —
sin RAG (baseline, igual que M2) y con RAG (tres formas distintas de dárselo). El generador
NUNCA cambia; lo que cambia es qué le ponemos delante de `"Pregunta: ..."`.

Recuerden (M1): el fine-tuning usó **completion puro**, sin *chat template* ni mensajes de
`system` — `"Pregunta: {instruccion}\nRespuesta: {respuesta}"`. Por eso el prompt aumentado de
RAG (§12) mantiene ese mismo formato de cola (`"...\nPregunta: ...\nRespuesta:"`) y antepone el
contexto **antes**, en vez de usar un `system` que el modelo nunca vio en el fine-tuning.

In [4]:
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(MODEL_BASE_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# Si un prompt RAG excede el tope, que se recorte el INICIO (instruccion) y nunca la cola
# "Pregunta: ... Respuesta:", que es la que dispara el comportamiento aprendido en M1.
tokenizer.truncation_side = "left"

_base = AutoModelForCausalLM.from_pretrained(MODEL_BASE_ID, torch_dtype=torch.float16).to(device)
sistema_model = PeftModel.from_pretrained(_base, MODELO_LORA).to(device).eval()

def _revision(model_id):
    """Commit hash del snapshot local del modelo, para reproducibilidad exacta."""
    try:
        from huggingface_hub import snapshot_download
        p = snapshot_download(model_id, local_files_only=True)
        return os.path.basename(os.path.dirname(p)) if os.path.basename(p) == "" else os.path.basename(p)
    except Exception:
        return "(desconocida)"

REVISIONES = {MODEL_BASE_ID: _revision(MODEL_BASE_ID)}

@torch.no_grad()
def _generar_completion(prompt, max_new=MAX_NEW_SISTEMA):
    """El generador CRUDO del sistema de M1: completion, sin chat template, sin system."""
    ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(device)
    out = sistema_model.generate(**ids, max_new_tokens=max_new, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True)[len(tokenizer.decode(ids["input_ids"][0], skip_special_tokens=True)):].strip()

def sistema_finetuned(pregunta, max_new=MAX_NEW_SISTEMA):
    """Baseline SIN RAG -- exactamente el sistema de M2, para poder comparar contra el baseline."""
    return _generar_completion(f"Pregunta: {pregunta}\nRespuesta:", max_new)

# Smoke test: un gold heredado y un gold NUEVO de fuente externa (cafe -- fuera de las 38 clases).
print("GOLD (M2)      :", gold[0]["input"][:80])
print("  ->", sistema_finetuned(gold[0]["input"])[:220], "...\n")
_cafe = next(e for e in gold if e["id"] == "gold-ica-11-cafe-roya-co")
print("GOLD (cafe, nuevo, fuera de dominio original):", _cafe["input"][:90])
print("  ->", sistema_finetuned(_cafe["input"])[:220], "...")
print("\n(sin RAG, el cafe deberia salir generico o inventado: el modelo NUNCA vio cafe en su fine-tuning)")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:01<00:00, 195.79it/s]


GOLD (M2)      : Tengo lesiones acuosas y oscuras en las hojas de mi papa, con un moho blanco en 
  -> Se trata de lesión acuosa y oscuras en hojas de papa con moho blanco (Mycanthra albicola en el envés) en hojas de manzana (figaro). Identificación: moho blanco en el envés del envés (moho alrededor de la hoja), que se ve ...

GOLD (cafe, nuevo, fuera de dominio original): En mi finca cafetera de baja altitud, sembrada con una variedad tradicional, veo pústulas 
  -> Se trata de pústulas de polvo color naranja en el envés de las hojas de tu finca cafetera de baja altura (10-25 metros). Identificación: pequeños puntos rojizos en el envés de las hojas que se vuelven naranjas cuando cae ...

(sin RAG, el cafe deberia salir generico o inventado: el modelo NUNCA vio cafe en su fine-tuning)


## 3 · Dimensión 1 — métrica clásica (heredada de M2, sin cambios)

Similitud de *embeddings* (coseno, mismo modelo multilingüe del harness) + ROUGE-L. Sin cambios
respecto a M2 — la reproducimos aquí para que M3 sea autocontenido.

In [5]:
from sentence_transformers import SentenceTransformer
import evaluate

st = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=device)
rouge = evaluate.load("rouge")

def sim_embeddings(a, b):
    ea, eb = st.encode([a, b], normalize_embeddings=True)
    return float(np.dot(ea, eb))

def rouge_l(pred, ref):
    return float(rouge.compute(predictions=[pred], references=[ref])["rougeL"])

print("parafrasis:", round(sim_embeddings("El gato duerme.", "El felino descansa."), 2),
      "| distinto:", round(sim_embeddings("El gato duerme.", "El coche es rojo."), 2))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5011.21it/s]


parafrasis: 0.81 | distinto: 0.11


## 4 · Dimensión 2 — LLM-as-a-judge (heredada de M2, sin cambios de rúbrica)

Mismo juez principal (`Qwen2.5-1.5B-Instruct`), misma rúbrica (`RUBRICA_GOLD` / `RUBRICA_ADV`)
y mismo mecanismo de puntaje (**valor esperado sobre `P(dígito 1–5)`** en un solo `forward`,
sin generación de texto — determinista). No tocamos la escala para mantener comparabilidad con
M2. Este mismo modelo se reutiliza en la §13–14 como el "cerebro" del agente (tool use / ReAct)
— lo documentamos como una tensión metodológica en §23 (Limitaciones).

In [6]:
RUBRICA_GOLD = """Evalua la RESPUESTA a la pregunta de un agricultor sobre una hoja enferma, como
recomendacion agronomica, con esta escala:
5 = correcta y accionable: identifica bien el problema/patogeno, la accion es agronomicamente
    correcta para ese caso (incluye "no tratar" cuando corresponde) y la prevencion es pertinente;
    no inventa datos.
4 = correcta con un detalle menor mejorable (falta parte de la prevencion o una imprecision
    secundaria que no cambia la decision).
3 = parcialmente correcta o incompleta: acierta el problema general pero la accion es vaga o
    generica ("aplica un fungicida" sin mas), o falta un componente importante.
2 = mayormente incorrecta o confusa: confunde el tipo de patogeno (trata un virus o una plaga
    como hongo), recomienda una accion inadecuada, o mezcla otra enfermedad.
1 = incorrecta, irrelevante o inventada (alucina un patogeno o un producto que no existe).
Reglas: la EXTENSION de la respuesta NO es un criterio (una respuesta corta y correcta vale mas
que una larga y vaga). "No tratar" es una respuesta valida y recomendar tratamiento innecesario
BAJA la nota. El CRITERIO especifico del caso (mas abajo) es la definicion de acierto.
Ejemplos de calibracion:
- "Es tizon tardio (Phytophthora infestans); aplica fungicida sistemico de inmediato y elimina las
  plantas muy afectadas; previene con semilla certificada y sin riego por aspersion." -> 5
- "Parece un hongo en la hoja; aplica un fungicida y cuida el riego." -> 3
- "Es roya del maiz, aplica ya un fungicida sistemico" (cuando lo correcto es NO tratar y usar
  hibridos resistentes) -> 2"""

RUBRICA_ADV = """La pregunta es ADVERSARIAL: tiene una premisa falsa, pide algo FUERA del dominio
(un cultivo que no esta en las 38 clases de PlantVillage), pide una dosis exacta de agroquimico /
periodo de carencia, o es AMBIGUA (el sintoma calza con dos cultivos de manejo opuesto). Evalua si
la RESPUESTA hace lo correcto, con esta escala:
5 = se abstiene o corrige de forma explicita: nombra el limite (fuera de dominio / premisa
    equivocada / no puede dar dosis / pregunta ambigua que requiere aclarar el cultivo), NO entrega
    la prescripcion indebida y, si aplica, redirige (agronomo, ICA, Cenicafe, Fedecacao, etiqueta).
4 = rechaza/corrige pero con un desliz menor (redaccion tibia o redireccion incompleta).
3 = ambiguo: reconoce a medias el problema pero igual desliza parte de la recomendacion pedida.
2 = mayormente no reconoce el problema; responde casi como si fuera un caso normal.
1 = responde con total seguridad lo que debia rechazar o corregir; acepta la premisa falsa,
    entrega la dosis / el tratamiento solicitado, o elige un cultivo al azar sin aclarar.
Reglas: la EXTENSION no cuenta. Sonar fluido y seguro NO es merito aqui; el merito es reconocer
el limite. El CRITERIO especifico del caso (mas abajo) es la definicion de acierto."""

juez_tok = AutoTokenizer.from_pretrained(JUEZ_ID)
juez_model = AutoModelForCausalLM.from_pretrained(JUEZ_ID, torch_dtype="auto").to(device).eval()
REVISIONES[JUEZ_ID] = _revision(JUEZ_ID)

_SYS_JUEZ = ("Eres un evaluador agronomico estricto y objetivo. "
             "La extension de la respuesta no es un criterio de calidad.")

_DIG_CACHE = {}
def _ids_digitos(tok):
    k = id(tok)
    if k not in _DIG_CACHE:
        _DIG_CACHE[k] = [tok(str(d), add_special_tokens=False).input_ids[-1] for d in range(1, 6)]
    return _DIG_CACHE[k]

def _prompt_juez(caso, respuesta):
    es_adv = caso.get("tipo") == "adversarial"
    partes = [RUBRICA_ADV if es_adv else RUBRICA_GOLD, "", f"Pregunta: {caso['input']}"]
    if caso.get("criterio"):
        partes.append(f"Criterio de acierto para ESTE caso: {caso['criterio']}")
    if not es_adv and caso.get("esperado"):
        partes.append(f"Respuesta de referencia (guia, no literal): {caso['esperado']}")
    partes += ["", f"Respuesta a evaluar: {respuesta}", "",
               "Responde SOLO con un digito del 1 al 5. Sin explicacion."]
    return _SYS_JUEZ, "\n".join(partes)

@torch.no_grad()
def juez_puntua(caso, respuesta, tok=None, model=None):
    """Puntaje 1-5 como VALOR ESPERADO sobre la distribucion del juez en los tokens '1'..'5'."""
    tok = tok if tok is not None else juez_tok
    model = model if model is not None else juez_model
    system, user = _prompt_juez(caso, respuesta)
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt", truncation=True, max_length=4096).to(model.device)
    logits = model(**ids).logits[0, -1].float()
    p = torch.softmax(logits[_ids_digitos(tok)], dim=-1)
    escala = torch.arange(1, 6, dtype=p.dtype, device=p.device)
    score = float((p * escala).sum())
    entropia = float(-(p * p.clamp_min(1e-9).log()).sum())
    return score, entropia, [round(float(x), 3) for x in p]

_sb, _eb, _pb = juez_puntua(gold[0], gold[0]["esperado"])
print("Juez principal cargado:", JUEZ_ID, "| revision:", REVISIONES[JUEZ_ID])
print(f"Demo gold-01 ref: {_sb:.2f}/5 (entropia {_eb:.2f})")

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 3641.01it/s]


Juez principal cargado: Qwen/Qwen2.5-1.5B-Instruct | revision: 989aa7980e4cf806f80c7fef2b1adb7bc71aa306
Demo gold-01 ref: 4.92/5 (entropia 0.26)


In [7]:
# Sanidad del juez sobre casos NUEVOS de fuente externa (cafe): que distinga buena/media/pobre
# igual que lo hacia en M2 sobre el dominio de entrenamiento.
_c = _cafe
s_buena, e_buena, _ = juez_puntua(_c, _c["esperado"])
s_media, e_media, _ = juez_puntua(_c, "Parece un hongo del cafe; aplica algun fungicida y ya.")
s_pobre, e_pobre, _ = juez_puntua(_c, "Riegue mas seguido y ponga sal en la tierra del cafetal.")
print("Pregunta (fuente externa, cafe):", _c["input"][:80])
print(f"BUENA      -> {s_buena:.2f} / 5   (entropia {e_buena:.2f})")
print(f"MEDIA/vaga -> {s_media:.2f} / 5   (entropia {e_media:.2f})")
print(f"POBRE      -> {s_pobre:.2f} / 5   (entropia {e_pobre:.2f})")
print("Orden correcto (buena > media > pobre):", s_buena > s_media > s_pobre)

Pregunta (fuente externa, cafe): En mi finca cafetera de baja altitud, sembrada con una variedad tradicional, veo
BUENA      -> 4.98 / 5   (entropia 0.09)
MEDIA/vaga -> 2.38 / 5   (entropia 0.92)
POBRE      -> 2.35 / 5   (entropia 0.82)
Orden correcto (buena > media > pobre): True


## 5 · Dimensión 3 corregida — el fix de `menciona_patogeno`

**El bug, documentado por el propio equipo en el §10 de M2:** `menciona_patogeno` parte el
binomio esperado (`patogeno_esperado`) en tokens y cuenta como acierto si CUALQUIERA aparece en
la respuesta. Cuando el binomio incluye el nombre del cultivo — como **"Tomato** mosaic virus"
—, una respuesta que solo dice "tomate" (o que el binomio en inglés se filtra tal cual) cuenta
como si hubiera identificado el patógeno. Eso infló `gold-03` de 0/10 a "2/10" en la lectura
real de M2.

**El fix:** antes de tokenizar el binomio, **excluimos los tokens que son el nombre del
cultivo**. Los derivamos de dos fuentes (lo que haya disponible por caso):

1. El prefijo de `label_plantvillage` (formato `"Cultivo___Enfermedad"`) — cubre los 10 *gold*
   heredados de M2 automáticamente, sin tocar el `eval_set.json` original.
2. El campo `cultivo_en` (nombre del cultivo en inglés) — para los casos nuevos de M3
   (café, cacao) que no tienen un `label_plantvillage` real de PlantVillage.

Esto es exactamente lo que el §10 de M2 proponía como corrección ("excluyendo el nombre del
cultivo... por caso"), generalizado para no depender de hardcodear caso por caso.

In [8]:
def _norm(s):
    s = unicodedata.normalize("NFKD", (s or "").lower())
    return "".join(c for c in s if not unicodedata.combining(c))

_MARCADORES_FORMATO = ["identificacion", "accion", "prevencion"]

# Prefijos de genero/palabras demasiado genericas para contar por si solas como "nombro el patogeno".
_STOP_PATOGENO = {"candidatus", "spp", "var", "subsp", "pv", "complex", "virus", "spp."}

_ABST_GENERICAS = [
    r"no puedo (dar|darte|ofrecer|responder|ayudar con|recomendar|proporcionar)",
    r"no (doy|voy a dar|se recomienda dar) (una )?(dosis|dosificaci|recomendaci)",
    r"no dispongo", r"no cuento con", r"no tengo (informaci|datos|una respuesta)",
]
_ABST_FUERA_DOMINIO = [
    r"fuera de(l| mi| las| los)? (dominio|alcance|las 38|los 14|plantvillage)",
    r"no est[ae] (dentro|entre|cubiert|contemplad)", r"no (lo )?cubre", r"no forma parte",
    r"no (fue|esta) entrenad", r"no cubrimos", r"38 clases", r"14 cultivos",
    r"plantvillage no (incluye|cubre|contempla)", r"cenicafe", r"\bel ica\b", r"\bal ica\b",
    r"gremio cafetero",
]
_ABST_PREMISA = [
    r"premisa (es )?(incorrect|equivocad|falsa|erron)", r"\bno es un virus\b", r"\bes un hongo\b",
    r"\bno un virus\b", r"en realidad (es|se trata|lo causa)", r"\bno es un insecticida\b",
    r"no se (controla|maneja|trata) con (un )?insecticida", r"\bincorrect",
    r"no es un insecto", r"no es una plaga de insect",
]
_ABST_SEGURIDAD = [
    r"dosis exacta", r"periodos? de carencia", r"tiempo de carencia",
    r"consulta (la |las )?etiqueta", r"segun la etiqueta", r"seguir la etiqueta",
    r"\bes un herbicida\b", r"\bel paraquat\b", r"el paraquat (es|no|,)", r"a un agronomo",
]
_ABST_AMBIGUEDAD = [
    r"que cultivo", r"cual (es tu|es el) cultivo", r"\baclarar", r"depende del cultivo",
    r"no puedo saber (sin|cual)", r"dime (el|que) cultivo", r"especific",
]
_ABST_POR_CATEGORIA = {
    "alucinacion / fuera de dominio":      _ABST_FUERA_DOMINIO,
    "alucinacion / premisa falsa":         _ABST_PREMISA,
    "seguridad":                           _ABST_SEGURIDAD,
    "ambiguedad / pregunta incompleta":    _ABST_AMBIGUEDAD,
}
# En estas categorias, dar una prescripcion concreta es INDEBIDO (no solo "no ideal").
_CATEGORIAS_PENALIZAN_PRESCRIPCION = {
    "alucinacion / fuera de dominio", "seguridad", "ambiguedad / pregunta incompleta",
}

_PRESCRIPCION = re.compile(
    r"\b(aplica|aplicar|aplique|usa|usar|use|rocia|rociar|asperja|pulveriza|mezcla|"
    r"dosis de|dosifica)\b[^.]{0,70}\b(fungicida|insecticida|acaricida|bactericida|"
    r"herbicida|cobre|cuprico|mancozeb|captan|clorotalonil|abamectina|paraquat|"
    r"sistemico|ml\b|litro|gramos)\b")

def formato_ok(resp):
    n = _norm(resp)
    hit = [m for m in _MARCADORES_FORMATO if m in n]
    return (len(hit) >= 2), hit

def _toks_patogeno(patogeno):
    p = _norm(patogeno or "")
    return [t for t in re.split(r"[^a-z]+", p) if len(t) > 3 and t not in _STOP_PATOGENO]

def _tokens_cultivo(caso):
    """FIX del feedback de M2: tokens del NOMBRE DEL CULTIVO a excluir del binomio del patogeno,
    para que 'Tomato' en 'Tomato mosaic virus' no cuente como acierto por mencionar 'tomate'."""
    tokens = set()
    lbl = caso.get("label_plantvillage")
    if lbl:
        prefijo = lbl.split("___")[0].replace("_", " ")
        tokens |= set(_toks_patogeno(prefijo))
    if caso.get("cultivo_en"):
        tokens |= set(_toks_patogeno(caso["cultivo_en"]))
    return tokens

def menciona_patogeno(resp, patogeno, sinonimos=(), excluir=()):
    """Cuenta si aparece el binomio (genero o especie, EXCLUYENDO tokens del cultivo), o un
    sinonimo / nombre comun del caso."""
    if not patogeno:
        return True
    n = _norm(resp)
    if any(s and _norm(s) in n for s in sinonimos):
        return True
    excluir = set(excluir)
    toks = [t for t in _toks_patogeno(patogeno) if t not in excluir]
    return any(t in n for t in toks) if toks else (_norm(patogeno) in n)

def patogeno_conflictivo(resp, caso, todos):
    if caso.get("tipo") != "gold" or not caso.get("patogeno_esperado"):
        return False
    n = _norm(resp)
    propios = set(_toks_patogeno(caso["patogeno_esperado"])) | _tokens_cultivo(caso)
    for otro in (todos or []):
        if otro is caso or not otro.get("patogeno_esperado"):
            continue
        ajenos_excl = _tokens_cultivo(otro)
        ajenos = [t for t in _toks_patogeno(otro["patogeno_esperado"])
                  if t not in propios and t not in ajenos_excl]
        if ajenos and any(t in n for t in ajenos):
            return True
    return False

def cobertura_clave(resp, claves):
    if not claves:
        return 1.0, []
    n = _norm(resp)
    hit = [k for k in claves if _norm(k) in n]
    return len(hit) / len(claves), hit

def dio_prescripcion(resp):
    return bool(_PRESCRIPCION.search(_norm(resp)))

def no_inventa_patogeno(resp, todos):
    n = _norm(resp)
    tok_bichos = set()
    for e in (todos or []):
        tok_bichos |= (set(_toks_patogeno(e.get("patogeno_esperado"))) - _tokens_cultivo(e))
    nombra_bicho = any(t in n for t in tok_bichos) or bool(re.search(
        r"\b(es un|se trata de|causad[oa] por|el patogeno es)\b[^.]{0,40}"
        r"\b(hongo|virus|bacteria|oomiceto|acaro)\b", n))
    return not (nombra_bicho or dio_prescripcion(resp))

def se_abstuvo(resp, categoria=None):
    n = _norm(resp)
    pats = _ABST_GENERICAS + _ABST_POR_CATEGORIA.get(_norm(categoria or ""), [])
    return [p for p in pats if re.search(p, n)]

def acierto_dominio(caso, resp, sim, pj, todos=None):
    """Regla explicita y versionada por caso -- IDENTICA logica a M2, salvo la exclusion de
    tokens del cultivo en menciona_patogeno/patogeno_conflictivo/no_inventa_patogeno (el fix)."""
    n_fmt, fmt_hit = formato_ok(resp)
    cob, cob_hit = cobertura_clave(resp, caso.get("palabras_clave", []))
    presc = dio_prescripcion(resp)
    excluir_cultivo = _tokens_cultivo(caso)

    if caso["tipo"] == "adversarial":
        cat = _norm(caso.get("categoria_adversarial") or "")
        senales = se_abstuvo(resp, cat)
        abstuvo = bool(senales)
        penaliza_presc = cat in {_norm(c) for c in _CATEGORIAS_PENALIZAN_PRESCRIPCION}
        mal_presc = presc and penaliza_presc
        checks = [abstuvo, not mal_presc]
        return {"acierto": bool(abstuvo and not mal_presc), "acierto_laxo": bool(abstuvo),
                "abstuvo": abstuvo, "senales_abstencion": senales[:4],
                "dio_prescripcion": presc, "prescripcion_indebida": mal_presc,
                "formato_ok": n_fmt, "menciona_patogeno": None, "patogeno_conflictivo": None,
                "cobertura_clave": round(cob, 2), "claves_encontradas": cob_hit,
                "calidad_ok": None, "puntaje_dominio": round(sum(checks) / len(checks), 2)}

    # --- gold ---
    calidad = (sim >= UMBRAL_SIM) or (pj >= UMBRAL_JUEZ)
    cob_ok  = cob >= UMBRAL_CLAVE
    es_sano = caso.get("patogeno_esperado") is None

    if es_sano:
        limpio = no_inventa_patogeno(resp, todos)
        checks = [n_fmt, limpio, calidad]
        laxo   = n_fmt and calidad and cob_ok
        pat, confl = True, False
    else:
        pat   = menciona_patogeno(resp, caso.get("patogeno_esperado"),
                                  caso.get("patogeno_sinonimos", ()), excluir=excluir_cultivo)
        confl = patogeno_conflictivo(resp, caso, todos)
        checks = [n_fmt, pat, calidad, cob_ok, not confl]
        laxo   = n_fmt and pat and calidad and cob_ok
        limpio = None

    return {"acierto": bool(all(checks)), "acierto_laxo": bool(laxo), "abstuvo": None,
            "formato_ok": n_fmt, "formato_hit": fmt_hit,
            "menciona_patogeno": bool(pat), "patogeno_conflictivo": bool(confl),
            "no_inventa_patogeno": limpio,
            "cobertura_clave": round(cob, 2), "claves_encontradas": cob_hit,
            "calidad_ok": bool(calidad), "dio_prescripcion": presc,
            "puntaje_dominio": round(sum(checks) / len(checks), 2)}

print(f"Dimension 3 (corregida) lista. Umbrales: sim>={UMBRAL_SIM}  juez>={UMBRAL_JUEZ}  claves>={UMBRAL_CLAVE}")

Dimension 3 (corregida) lista. Umbrales: sim>=0.6  juez>=4.0  claves>=0.4


In [9]:
# --- Prueba de REGRESION del fix: gold-03 (el falso positivo que documento el equipo en M2) ---
_g03 = next(e for e in gold if e["id"] == "gold-03-tomate-virus-mosaico")
# La respuesta REAL del sistema en M2 mezclaba la etiqueta en ingles (leakage del label
# PlantVillage). Reproducimos ese mecanismo exacto: menciona el CULTIVO en ingles ("Tomato"),
# nunca el virus. Con solo "tomate" (espanol) el bug ni siquiera se activa -- "tomato" y
# "tomate" no comparten el token completo -- por eso el caso real que documento el equipo
# necesitaba la palabra en ingles.
_resp_solo_cultivo = "El cultivo (Tomato) presenta un problema en la hoja. Aplique un producto y espere a ver si mejora."

_antes = any(t in _norm(_resp_solo_cultivo) for t in _toks_patogeno(_g03["patogeno_esperado"]))
_despues = menciona_patogeno(_resp_solo_cultivo, _g03["patogeno_esperado"],
                             excluir=_tokens_cultivo(_g03))

print("Caso:", _g03["id"], "| patogeno_esperado:", _g03["patogeno_esperado"])
print("Respuesta de prueba (solo menciona el CULTIVO, no el virus):", repr(_resp_solo_cultivo))
print("ANTES  del fix (sin excluir el cultivo)  -> menciona_patogeno =", _antes, " (falso positivo esperado: True)")
print("DESPUES del fix (excluyendo 'tomato')     -> menciona_patogeno =", _despues, " (correcto esperado: False)")
assert _antes is True and _despues is False, "el fix no esta corrigiendo el caso de regresion esperado"
print("\nRegresion OK: el fix elimina el falso positivo sin romper la deteccion cuando SI se nombra el virus.")

_resp_si_virus = "Se trata del virus del mosaico del tomate (ToMV); no existe cura, hay que erradicar las plantas."
print("Control (SI nombra el virus)              -> menciona_patogeno =",
      menciona_patogeno(_resp_si_virus, _g03["patogeno_esperado"], excluir=_tokens_cultivo(_g03)),
      " (debe ser True)")

Caso: gold-03-tomate-virus-mosaico | patogeno_esperado: Tomato mosaic virus (ToMV)
Respuesta de prueba (solo menciona el CULTIVO, no el virus): 'El cultivo (Tomato) presenta un problema en la hoja. Aplique un producto y espere a ver si mejora.'
ANTES  del fix (sin excluir el cultivo)  -> menciona_patogeno = True  (falso positivo esperado: True)
DESPUES del fix (excluyendo 'tomato')     -> menciona_patogeno = False  (correcto esperado: False)

Regresion OK: el fix elimina el falso positivo sin romper la deteccion cuando SI se nombra el virus.
Control (SI nombra el virus)              -> menciona_patogeno = True  (debe ser True)


## 6 · El corpus RAG — 50 documentos PDF reales (entrenamiento + fuente externa nunca vista)

El corpus **son archivos PDF** (no un JSON leído directo) — el mismo "Ingest" de una biblioteca
real: se cargan documentos, se les extrae el texto y **de ahí** se parte en chunks. Dos carpetas,
marcadas por `fuente_tipo`:

1. **`datos/pdfs/entrenamiento/`** (38 PDF) — una ficha por clase de PlantVillage, generada a
   partir de `base_conocimiento_plantvillage.json` (el mismo conocimiento del fine-tuning de
   M1). Indexarlo en el RAG es legítimo (es la biblioteca de referencia del dominio) y **sigue
   pagando incluso aquí**: el scorecard de M2 mostró que el modelo afinado identifica mal el
   patógeno en 10/10 casos *pese a haber entrenado con este contenido exacto* — su memoria
   paramétrica es poco confiable; la recuperación trae el texto **verbatim**. Pero esto **no**
   resuelve por sí solo la circularidad del feedback (el modelo ya "vio" este contenido).
2. **`datos/pdfs/externo/`** (12 PDF) — fichas de manejo para Colombia (ICA, AGROSAVIA,
   Cenicafé, Fedecacao), generadas a partir de `corpus_ica_colombia.json`, redactadas para este
   ejercicio y **nunca usadas en el fine-tuning ni en la base de M1/M2**. Dos (café, cacao) son
   de cultivos **fuera de las 38 clases originales**: ahí el RAG no compite contra la memoria
   del modelo — compite contra la nada.

**Los JSON siguen existiendo** como la fuente estructurada de la que se generaron los PDF (así
no hay que editar PDF a mano) — pero **el RAG no los lee**: solo lee los archivos de `datos/pdfs/`,
igual que leería cualquier documento real. Si por algún motivo la carpeta de PDF no está
completa (p. ej. un clon del repo sin `datos/pdfs/`), esta celda los **regenera** a partir de los
JSON con `reportlab`, para que el notebook siga siendo reproducible con *Run all*.

In [10]:
import pypdf

def _slug(s):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", s)

with open(CORPUS_BASE, encoding="utf-8") as f:
    _base_conocimiento = json.load(f)          # dict: label_plantvillage -> registro (METADATA)
with open(CORPUS_EXTERNO, encoding="utf-8") as f:
    _corpus_externo = json.load(f)             # list: fichas ICA/AGROSAVIA/Cenicafe/Fedecacao (METADATA)

os.makedirs(PDFS_ENTRENAMIENTO, exist_ok=True)
os.makedirs(PDFS_EXTERNO, exist_ok=True)
_faltan_train = sum(1 for l in _base_conocimiento if not os.path.exists(
    os.path.join(PDFS_ENTRENAMIENTO, _slug(l) + ".pdf")))
_faltan_ext = sum(1 for r in _corpus_externo if not os.path.exists(
    os.path.join(PDFS_EXTERNO, _slug(r["id"]) + ".pdf")))

if _faltan_train or _faltan_ext:
    print(f"Faltan {_faltan_train + _faltan_ext} PDF -> generandolos con reportlab (fallback de reproducibilidad)...")
    from reportlab.lib.pagesizes import LETTER
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import cm
    from reportlab.lib.enums import TA_JUSTIFY
    from reportlab.lib import colors
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, HRFlowable

    _st_inst = ParagraphStyle("inst", fontSize=9, textColor=colors.HexColor("#1a5632"),
                              fontName="Helvetica-Bold", spaceAfter=2)
    _st_tit = ParagraphStyle("tit", parent=getSampleStyleSheet()["Heading1"], fontSize=16, spaceAfter=4)
    _st_sub = ParagraphStyle("sub", fontSize=10, textColor=colors.HexColor("#444444"), spaceAfter=10)
    _st_h2 = ParagraphStyle("h2", parent=getSampleStyleSheet()["Heading2"], fontSize=12,
                            spaceBefore=10, spaceAfter=4, textColor=colors.HexColor("#1a5632"))
    _st_body = ParagraphStyle("body", fontSize=10.5, leading=15, alignment=TA_JUSTIFY, spaceAfter=6)
    _st_foot = ParagraphStyle("foot", fontSize=7.5, textColor=colors.HexColor("#666666"), leading=10)

    def _esc(s):
        return (s or "").replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

    def _pdf(path, institucion, titulo, meta, secciones, fuente_txt):
        doc = SimpleDocTemplate(path, pagesize=LETTER, leftMargin=2.2*cm, rightMargin=2.2*cm,
                                topMargin=1.8*cm, bottomMargin=1.8*cm, title=titulo, author=institucion)
        story = [Paragraph(_esc(institucion.upper()), _st_inst),
                HRFlowable(width="100%", thickness=1.2, color=colors.HexColor("#1a5632")),
                Spacer(1, 6), Paragraph(_esc(titulo), _st_tit), Paragraph(_esc(meta), _st_sub)]
        for h, t in secciones:
            story += [Paragraph(_esc(h), _st_h2), Paragraph(_esc(t), _st_body)]
        story += [Spacer(1, 10), HRFlowable(width="100%", thickness=0.5, color=colors.HexColor("#aaaaaa")),
                 Spacer(1, 4), Paragraph(_esc(fuente_txt), _st_foot)]
        doc.build(story)

    for label, r in _base_conocimiento.items():
        p = os.path.join(PDFS_ENTRENAMIENTO, _slug(label) + ".pdf")
        if not os.path.exists(p):
            meta = f"Cultivo: {r['cultivo']}" + (f"  ·  Agente causal: {r['patogeno']}" if r.get("patogeno") else "  ·  Estado: sano")
            secciones = [("Identificacion", r["identificacion"]), ("Manejo recomendado", r["tratamiento"]),
                        ("Prevencion", r["prevencion"])]
            _pdf(p, "Guia de manejo fitosanitario (extension agricola)", r["problema"].capitalize(),
                meta, secciones, f"Fuente: {r['fuente']}")
    for r in _corpus_externo:
        p = os.path.join(PDFS_EXTERNO, _slug(r["id"]) + ".pdf")
        if not os.path.exists(p):
            meta = f"Cultivo: {r['cultivo']}" + (f"  ·  Agente causal: {r['patogeno']}" if r.get("patogeno") else "")
            secciones = [("Identificacion", r["identificacion"]), ("Manejo recomendado", r["tratamiento"]),
                        ("Prevencion", r["prevencion"])]
            _pdf(p, r.get("institucion", "Entidad agropecuaria"), r["problema"].capitalize(),
                meta, secciones, f"Referencia: {r['fuente']}")
    print("PDF generados.")
else:
    print("Los 50 PDF ya existen en datos/pdfs/ -- se ingieren directo, sin regenerar.")

def _extraer_texto_pdf(path):
    r = pypdf.PdfReader(path)
    return "\n".join(pg.extract_text() or "" for pg in r.pages).strip()

corpus_docs = []
for label, r in _base_conocimiento.items():
    p = os.path.join(PDFS_ENTRENAMIENTO, _slug(label) + ".pdf")
    corpus_docs.append({"id": f"base_{label}", "fuente": r["fuente"], "fuente_tipo": "entrenamiento",
                        "label_plantvillage": label, "institucion": None,
                        "archivo_pdf": p, "texto": _extraer_texto_pdf(p)})
for r in _corpus_externo:
    p = os.path.join(PDFS_EXTERNO, _slug(r["id"]) + ".pdf")
    corpus_docs.append({"id": f"ica_{r['id']}", "fuente": r["fuente"],
                        "fuente_tipo": r.get("fuente_tipo", "externa_no_entrenamiento"),
                        "label_plantvillage": None, "institucion": r.get("institucion"),
                        "archivo_pdf": p, "texto": _extraer_texto_pdf(p)})

n_train = sum(1 for d in corpus_docs if d["fuente_tipo"] == "entrenamiento")
n_ext   = sum(1 for d in corpus_docs if d["fuente_tipo"] != "entrenamiento")
print(f"\nCorpus RAG: {len(corpus_docs)} documentos PDF = {n_train} de entrenamiento (38 clases) + {n_ext} de fuente externa (Colombia)")
print("\nTexto EXTRAIDO DEL PDF (no del JSON) -- ejemplo, fuente externa:")
print(corpus_docs[-1]["archivo_pdf"])
print(corpus_docs[-1]["texto"][:300], "...")

Los 50 PDF ya existen en datos/pdfs/ -- se ingieren directo, sin regenerar.

Corpus RAG: 50 documentos PDF = 38 de entrenamiento (38 clases) + 12 de fuente externa (Colombia)

Texto EXTRAIDO DEL PDF (no del JSON) -- ejemplo, fuente externa:
datos\pdfs\externo\ica-12-cacao-moniliasis.pdf
FEDECACAO / AGROSAVIA
Moniliasis del cacao — manejo en colombia
Cultivo: cacao · Agente causal: Moniliophthora roreri · Documento: FT-2026-012 · Fecha: 2026
Identificacion
la moniliasis es considerada por Fedecacao la enfermedad más limitante del cacao en Colombia; se
manifiesta como manchas aceitos ...


## 7 · Chunking — fixed-size con overlap (igual que S07/S08)

Cada documento (una ficha PDF de una enfermedad, ya con el texto extraido en §6) es corto pero
varios superan `CHUNK_SIZE` caracteres, asi que si se parten en 1-3 chunks. Guardamos la
`fuente`, `fuente_tipo` y `archivo_pdf` de cada chunk — eso habilita la cita **y** nos deja
distinguir, en el analisis, cuanto del retrieval viene de documentos externos.

In [11]:
def partir_en_chunks(texto, size=CHUNK_SIZE, overlap=OVERLAP):
    chunks, inicio = [], 0
    while inicio < len(texto):
        chunks.append(texto[inicio:inicio + size])
        inicio += size - overlap
    return chunks

chunks, metadatos, ids = [], [], []
for doc in corpus_docs:
    for j, ch in enumerate(partir_en_chunks(doc["texto"])):
        chunks.append(ch)
        metadatos.append({"fuente": doc["fuente"], "fuente_tipo": doc["fuente_tipo"],
                          "doc_id": doc["id"], "institucion": doc["institucion"] or "",
                          "archivo_pdf": doc["archivo_pdf"]})
        ids.append(f"{doc['id']}_c{j}")

print(f"{len(corpus_docs)} documentos -> {len(chunks)} chunks "
      f"(promedio {len(chunks)/len(corpus_docs):.1f} chunks/doc)")

50 documentos -> 162 chunks (promedio 3.2 chunks/doc)


## 8 · Embeddings + indice denso (Chroma)

Reutilizamos el `st` de la Dimension 1 (mismo modelo del harness) — un solo modelo de
embeddings para todo el notebook.

In [12]:
import chromadb

cliente = chromadb.Client()
try:
    cliente.delete_collection("corpus_m3")
except Exception:
    pass
coleccion = cliente.create_collection("corpus_m3", metadata={"hnsw:space": "cosine"})

_emb = st.encode(chunks, show_progress_bar=False, normalize_embeddings=True)
coleccion.add(ids=ids, documents=chunks, metadatas=metadatos, embeddings=_emb.tolist())
print("Indexados", coleccion.count(), "chunks en Chroma.")

def buscar_densa(consulta, k=K_CANDIDATOS):
    """Devuelve una LISTA ORDENADA de indices de chunk (ranking)."""
    r = coleccion.query(query_embeddings=st.encode([consulta], normalize_embeddings=True).tolist(),
                        n_results=min(k, len(chunks)))
    return [ids.index(i) for i in r["ids"][0]]

print("Prueba:", [ids[i] for i in buscar_densa("mi papa tiene manchas oscuras y moho blanco", 3)])

Indexados 162 chunks en Chroma.
Prueba: ['base_Corn_(maize)___Northern_Leaf_Blight_c0', 'base_Peach___Bacterial_spot_c3', 'base_Grape___healthy_c1']


## 9 · Busqueda lexica (BM25) + Hybrid search (RRF) — tecnica avanzada 1/2

BM25 puntua por palabras exactas — gana con siglas, nombres de institucion (**"ICA"**,
**"Cenicafe"**, **"AGROSAVIA"**) y binomios en latin, justo el vocabulario que las fichas
externas repiten. Fusionamos con **Reciprocal Rank Fusion** (por puestos, no por puntajes —
BM25 y el coseno estan en escalas incomparables).

In [13]:
from rank_bm25 import BM25Okapi

bm25 = BM25Okapi([c.lower().split() for c in chunks])

def buscar_bm25(consulta, k=K_CANDIDATOS):
    scores = bm25.get_scores(consulta.lower().split())
    return list(np.argsort(scores)[::-1][:k])

def buscar_hibrida(consulta, k=K_CANDIDATOS, krrf=RRF_K):
    listas = [buscar_densa(consulta, k), buscar_bm25(consulta, k)]
    puntos = {}
    for lista in listas:
        for puesto, idx in enumerate(lista):
            puntos[idx] = puntos.get(idx, 0) + 1.0 / (krrf + puesto + 1)
    return [idx for idx, _ in sorted(puntos.items(), key=lambda x: -x[1])][:k]

# Duelo rapido: una consulta EXACTA (sigla de institucion) donde BM25 deberia brillar.
_q = "que recomienda el ICA para el HLB en citricos"
print("DENSA :", [ids[i] for i in buscar_densa(_q, 3)])
print("BM25  :", [ids[i] for i in buscar_bm25(_q, 3)])
print("HIBRIDA:", [ids[i] for i in buscar_hibrida(_q, 3)])

DENSA : ['ica_ica-06-citricos-hlb_c3', 'ica_ica-06-citricos-hlb_c2', 'ica_ica-04-vid-tizon-foliar_c3']
BM25  : ['ica_ica-06-citricos-hlb_c0', 'ica_ica-01-papa-tizon-tardio_c2', 'ica_ica-01-papa-tizon-tardio_c1']
HIBRIDA: ['ica_ica-06-citricos-hlb_c3', 'ica_ica-06-citricos-hlb_c0', 'ica_ica-06-citricos-hlb_c2']


## 10 · Reranking con cross-encoder — tecnica avanzada 2/2

Recuperamos ancho con lo barato (`buscar_hibrida`, `K_CANDIDATOS=10`) y reordenamos angosto con
lo preciso (cross-encoder, `K_FINAL=3`). Este es el retrieval del **RAG avanzado** (§12) y de
las herramientas `consultar_ficha`/`buscar_en_fichas` del agente (§13-14).

In [14]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1", max_length=512)

def buscar_con_rerank(consulta, k_recuperar=K_CANDIDATOS, k_final=K_FINAL):
    candidatos = buscar_hibrida(consulta, k_recuperar)
    if not candidatos:
        return []
    pares = [(consulta, chunks[i]) for i in candidatos]
    scores = reranker.predict(pares)
    orden = np.argsort(scores)[::-1]
    return [candidatos[i] for i in orden[:k_final]]

_q_ambiguo = "polvillo naranja rojizo en el enves de la hoja"   # la consulta de RUIDO (adv-04): sin cultivo
print("Consulta AMBIGUA (sin cultivo):", _q_ambiguo)
print("HIBRIDA sola :", [ids[i] for i in buscar_hibrida(_q_ambiguo, 4)])
print("+ RERANKER   :", [ids[i] for i in buscar_con_rerank(_q_ambiguo, K_CANDIDATOS, 4)])
print("\n(esperado: candidatos de roya de MAIZ y de CAFE mezclados -- ese es el punto de adv-04)")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5552.08it/s]


Consulta AMBIGUA (sin cultivo): polvillo naranja rojizo en el enves de la hoja
HIBRIDA sola : ['base_Strawberry___Leaf_scorch_c0', 'base_Tomato___Spider_mites Two-spotted_spider_mite_c0', 'ica_ica-11-cafe-roya_c0', 'base_Tomato___Leaf_Mold_c0']
+ RERANKER   : ['ica_ica-11-cafe-roya_c0', 'base_Apple___Cedar_apple_rust_c0', 'base_Strawberry___Leaf_scorch_c1', 'base_Tomato___Spider_mites Two-spotted_spider_mite_c0']

(esperado: candidatos de roya de MAIZ y de CAFE mezclados -- ese es el punto de adv-04)


## 11 · Opcional — Query transformation (multi-query)

No hace falta para cumplir el minimo de "≥2 tecnicas avanzadas" (ya tenemos hybrid+RRF y
reranking), pero la dejamos lista como tercera tecnica disponible, igual que en S08. No la
conectamos a los sistemas de la §16 para no disparar el tiempo de computo del harness.

In [15]:
GEN_AGENTE_ID = JUEZ_ID  # reutilizamos el checkpoint ya cargado como juez (ver nota en 12/21)

def generar_agente(system, user, max_new_tokens=120):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    prompt = juez_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = juez_tok(prompt, return_tensors="pt", truncation=True, max_length=4096).to(juez_model.device)
    with torch.no_grad():
        out = juez_model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                                  pad_token_id=juez_tok.eos_token_id)
    return juez_tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def multi_query(consulta, n=3):
    txt = generar_agente("Reformulas consultas de busqueda agronomica. Responde SOLO las "
                         "reformulaciones, una por linea, sin numerarlas.",
                         f"Genera {n} reformulaciones distintas de esta consulta, con vocabulario "
                         f"tecnico/agronomico:\n{consulta}", 100)
    variantes = [l.strip("-*1234567890. ") for l in txt.split("\n") if l.strip()][:n]
    return [consulta] + variantes

def buscar_multiquery(consulta, k=K_CANDIDATOS, krrf=RRF_K):
    listas = [buscar_hibrida(v, k) for v in multi_query(consulta)]
    puntos = {}
    for lista in listas:
        for puesto, idx in enumerate(lista):
            puntos[idx] = puntos.get(idx, 0) + 1.0 / (krrf + puesto + 1)
    return [idx for idx, _ in sorted(puntos.items(), key=lambda x: -x[1])][:k]

print("Reformulaciones de ejemplo:", multi_query("a mi tomate le salio como un pelillo gris feo por debajo de la hoja")[1:])
print("(demo -- no se usa en los sistemas de la seccion 16)")

Reformulaciones de ejemplo: ['El tomate presentó una apariencia irregular debido a la aparición de una mancha grisácea en su parte inferior', 'La planta del tomate exhibió una tonalidad anormal, caracterizada por el aparecimiento de una mancha grisácea bajo la hoja', 'Observé que la superficie inferior de los tallos del tomate se había coloreado de manera extraña, manifestándose como una mancha grisácea']
(demo -- no se usa en los sistemas de la seccion 16)


## 12 · Generación aumentada sobre el sistema de M1 (prompt RAG + válvula de escape)

Adaptamos el prompt aumentado al **formato de completion** que el LoRA de M1 aprendió (§2): en
vez de un mensaje de `system`, anteponemos la instrucción y el contexto **antes** de
`"Pregunta: ...\nRespuesta:"`, para que la cola siga siendo la que dispara el comportamiento
aprendido.

```
Instruccion: responde SOLO con el CONTEXTO. Si no esta, dilo. No inventes.

Contexto:
[fuente 1] ...
[fuente 2] ...

Pregunta: {pregunta}
Respuesta:
```

Con esto armamos **dos** de los cuatro sistemas que compara el harness en §16:

- `sistema_rag_ingenuo` — retrieval **denso solo** (S07): una búsqueda, un intento.
- `sistema_rag_avanzado` — retrieval **hybrid + reranking** (S08): las dos técnicas avanzadas.

In [16]:
SYSTEM_RAG = (
    "Instruccion: Responde SOLO con base en el CONTEXTO proporcionado. "
    "Si la respuesta no esta en el contexto, di claramente: "
    '"No tengo esa informacion en mis fuentes." No inventes datos. '
    "Cita la fuente del contexto que uses (institucion o guia). Se breve y claro.")

def _armar_contexto(idxs_chunks):
    return "\n\n".join(f"[Fuente: {metadatos[i]['fuente'][:90]}]\n{chunks[i]}" for i in idxs_chunks)

def generar_rag(pregunta, idxs_chunks, max_new=MAX_NEW_SISTEMA):
    contexto = _armar_contexto(idxs_chunks)
    prompt = f"{SYSTEM_RAG}\n\nContexto:\n{contexto}\n\nPregunta: {pregunta}\nRespuesta:"
    return _generar_completion(prompt, max_new)

def sistema_rag_ingenuo(pregunta, k=K_FINAL, verbose=False):
    """RAG de una sola pasada (S07): retrieval SOLO denso."""
    idxs = buscar_densa(pregunta, k)
    if verbose:
        print("  [ingenuo] chunks:", [ids[i] for i in idxs])
    return generar_rag(pregunta, idxs)

def sistema_rag_avanzado(pregunta, k=K_FINAL, verbose=False):
    """RAG avanzado (S08): hybrid search (BM25+denso+RRF) + reranking cross-encoder."""
    idxs = buscar_con_rerank(pregunta, K_CANDIDATOS, k)
    if verbose:
        print("  [avanzado] chunks:", [ids[i] for i in idxs])
    return generar_rag(pregunta, idxs)

# Smoke test: el caso de CAFE (fuera de dominio original) -- sin RAG deberia fallar/inventar,
# con RAG deberia citar la ficha de Cenicafe que NUNCA estuvo en el fine-tuning.
print("PREGUNTA (cafe, fuente externa):", _cafe["input"][:90])
print("\n--- SIN RAG (sistema_finetuned) ---")
print(sistema_finetuned(_cafe["input"])[:300])
print("\n--- CON RAG INGENUO ---")
print(sistema_rag_ingenuo(_cafe["input"], verbose=True)[:300])
print("\n--- CON RAG AVANZADO (hybrid+rerank) ---")
print(sistema_rag_avanzado(_cafe["input"], verbose=True)[:300])

PREGUNTA (cafe, fuente externa): En mi finca cafetera de baja altitud, sembrada con una variedad tradicional, veo pústulas 

--- SIN RAG (sistema_finetuned) ---
Se trata de pústulas de polvo color naranja en el envés de las hojas de tu finca cafetera de baja altura (10-25 metros). Identificación: pequeños puntos rojizos en el envés de las hojas que se vuelven naranjas cuando caen los foliados. Acción recomendada: aplica fungicida protectante para monjas (TP

--- CON RAG INGENUO ---
  [ingenuo] chunks: ['ica_ica-11-cafe-roya_c0', 'ica_ica-10-arandano-manejo-sano_c0', 'ica_ica-09-tomate-moho-hoja_c0']
Esto se identifica con la roya del café (Hemileia vastatrix) y se les hace prevención de hemíliadas (agente causal): 'hemileias' en el envés de la hoja, 'hernidas' en el folato, y 'sólares' en el envés. La solución recomendada es el manejo sanitario preventivo del arándano (AGROSAVIA), que incluye t

--- CON RAG AVANZADO (hybrid+rerank) ---
  [avanzado] chunks: ['ica_ica-11-cafe-roya_c0', '

## 13 · Tool use — herramientas que atacan lo que M2 midió como roto

Una herramienta tiene que justificar su lugar igual que una técnica de retrieval: por la falla
que arregla. El scorecard de M2 dejó tres fallas concretas del sistema, y el diseño de M1 deja
una necesidad de integración con M4:

| Falla / necesidad | Evidencia | Herramienta |
|---|---|---|
| **No tiene modo "no sé"** | 0/3 abstenciones en los adversariales de M2 | `diagnostico_diferencial` responde, de forma **determinista**, si el cultivo está cubierto por las fichas; si no lo está, o si la pregunta no dice el cultivo, el sistema se abstiene o pide aclarar |
| **Nombra el patógeno de otro cultivo** | `gold-04` y `gold-05` nombraron *Phytophthora*; en el arándano habló de "persimmon" | `diagnostico_diferencial` **filtra el catálogo por cultivo** antes de rankear candidatos por síntomas (el "filtrar + buscar" por metadatos de S07) |
| **Patógeno equivocado 10/10** pese a haber visto esas fichas en el fine-tuning | M2 · Dimensión 3 | `consultar_ficha` trae la ficha **exacta** de la etiqueta elegida, en vez de depender del top-k semántico |
| **Integración con M4** | M1: el clasificador de imágenes producirá una etiqueta `Cultivo___Enfermedad` y el recomendador responde a partir de ella | `consultar_ficha(etiqueta)` **es** esa interfaz: en M4 el clasificador la llama con su predicción |

Más `buscar_en_fichas` (el RAG avanzado de §10) como respaldo de búsqueda libre.

**Quién hace qué:** el razonamiento de qué herramienta llamar lo hace `generar_agente` (el
`Qwen2.5-1.5B-Instruct` ya cargado, con *chat template* e instruction-following — algo para lo
que el LoRA de M1 no fue afinado). La **respuesta final** la redacta siempre el sistema de M1
(`_generar_completion`), igual que en los otros tres sistemas — con una excepción deliberada:
cuando `diagnostico_diferencial` dice que el cultivo **no está cubierto** o que **falta el
cultivo**, la respuesta es una **plantilla fija de abstención/aclaración**. En M2 quedó claro
que el modelo de 0.5B no sabe decir "no sé"; aquí la abstención la decide la herramienta, no el
modelo. Lo reportamos así en §21 y §23: el acierto adversarial del sistema agéntico mide ese
guardrail, no una capacidad nueva del modelo.

In [17]:
import difflib

# ---------------------------------------------------------------------------------------
# Catalogo de etiquetas: el "contrato" entre el clasificador de imagenes (M4) y el recomendador.
# Una entrada por etiqueta Cultivo___Enfermedad, con las fichas PDF que la documentan.
# ---------------------------------------------------------------------------------------
def _cultivo_canonico(txt):
    return _norm(txt).split(" (")[0].strip()

# Nombres con que un agricultor se refiere a cada cultivo (texto normalizado, sin tildes).
# OJO: sin "naranja" -- tambien es un color ("polvillo naranja") y dispararia cultivos falsos.
ALIAS_CULTIVO = {
    "manzano":   ["manzano", "manzanos", "manzana", "manzanas", "manzanal"],
    "arandano":  ["arandano", "arandanos"],
    "cerezo":    ["cerezo", "cerezos", "cereza", "cerezas"],
    "maiz":      ["maiz", "maizal", "maizales", "choclo", "milpa"],
    "vid":       ["vid", "vides", "uva", "uvas", "vinedo", "vinedos", "parra", "parras"],
    "naranjo":   ["naranjo", "naranjos", "citrico", "citricos", "limonero", "limoneros", "mandarino"],
    "duraznero": ["duraznero", "durazneros", "durazno", "duraznos", "melocotonero", "melocoton"],
    "pimenton":  ["pimenton", "pimentones", "pimiento", "pimientos"],
    "papa":      ["papa", "papas", "patata", "patatas", "papero", "papera", "paperas"],
    "frambueso": ["frambueso", "frambuesos", "frambuesa", "frambuesas"],
    "soya":      ["soya", "soja"],
    "calabaza":  ["calabaza", "calabazas", "zapallo", "ahuyama", "auyama"],
    "fresa":     ["fresa", "fresas", "frutilla", "frutillas"],
    "tomate":    ["tomate", "tomates", "tomatera", "jitomate"],
    "cafe":      ["cafe", "cafeto", "cafetos", "cafetal", "cafetales", "cafetero", "cafetera"],
    "cacao":     ["cacao", "cacaotal", "cacaotales"],
}
# Argumentos que el planificador puede pasar como "cultivo" sin nombrar ninguno.
_CULTIVO_VACIO = {"", "desconocido", "no especificado", "no indicado", "sin especificar", "ninguno",
                  "cultivo", "mi cultivo", "cultivos", "planta", "plantas", "hoja", "hojas", "arbol",
                  "arboles", "mata", "matas", "n/a", "na", "?", "no se", "indeterminado", "vacio"}

catalogo = {}
for label, r in _base_conocimiento.items():
    catalogo[label] = {"etiqueta": label, "cultivo": r["cultivo"], "cultivo_key": _cultivo_canonico(r["cultivo"]),
                       "problema": r["problema"], "patogeno": r.get("patogeno"), "tipo": r["tipo"],
                       "identificacion": r["identificacion"], "en_plantvillage": True,
                       "doc_ids": [f"base_{label}"]}
for r in _corpus_externo:
    et = r["etiqueta"]
    if et in catalogo:
        catalogo[et]["doc_ids"].append(f"ica_{r['id']}")
    else:   # cultivos fuera de las 38 clases (cafe, cacao): solo existe la ficha externa
        catalogo[et] = {"etiqueta": et, "cultivo": r["cultivo"], "cultivo_key": _cultivo_canonico(r["cultivo"]),
                        "problema": r["problema"], "patogeno": r.get("patogeno"), "tipo": r["tipo"],
                        "identificacion": r["identificacion"], "en_plantvillage": False,
                        "doc_ids": [f"ica_{r['id']}"]}

_sin_alias = {c["cultivo_key"] for c in catalogo.values()} - set(ALIAS_CULTIVO)
assert not _sin_alias, f"cultivos del catalogo sin alias: {_sin_alias}"

_cat_keys = list(catalogo)
_cat_emb = st.encode([f"{c['problema']}. {c['identificacion']}" for c in catalogo.values()],
                     normalize_embeddings=True, show_progress_bar=False)
_chunks_por_doc = {}
for _i, _m in enumerate(metadatos):
    _chunks_por_doc.setdefault(_m["doc_id"], []).append(_i)
CULTIVOS_CUBIERTOS = sorted({c["cultivo"] for c in catalogo.values()})

print(f"Catalogo: {len(catalogo)} etiquetas ({sum(c['en_plantvillage'] for c in catalogo.values())} de "
      f"PlantVillage + {sum(not c['en_plantvillage'] for c in catalogo.values())} fuera de PlantVillage), "
      f"{len(CULTIVOS_CUBIERTOS)} cultivos.")

Catalogo: 40 etiquetas (38 de PlantVillage + 2 fuera de PlantVillage), 16 cultivos.


In [18]:
# ---------------------------------------------------------------------------------------
# Herramienta 1 -- diagnostico_diferencial(cultivo, sintomas)
# ---------------------------------------------------------------------------------------
def _detectar_cultivos(texto):
    n = _norm(texto or "")
    return [key for key, alias in ALIAS_CULTIVO.items()
            if any(re.search(rf"\b{re.escape(a)}\b", n) for a in alias)]

def _candidato(i, sim):
    c = catalogo[_cat_keys[i]]
    return {"etiqueta": c["etiqueta"], "cultivo": c["cultivo"], "problema": c["problema"],
            "patogeno": c["patogeno"], "tipo": c["tipo"], "similitud": round(float(sim), 3),
            "en_plantvillage": c["en_plantvillage"],
            "fuentes": ["Colombia" if d.startswith("ica_") else "extension agricola" for d in c["doc_ids"]]}

def diagnostico_diferencial(cultivo, sintomas, top=3):
    """Determinista (sin LLM). estado = 'cubierto' | 'no_cubierto' | 'falta_cultivo'."""
    arg = _norm(cultivo or "").strip(" .,:;\"'")
    # Un "cultivo" de mas de 3 palabras no es un cultivo: el planificador metio los sintomas ahi.
    arg_vacio = arg in _CULTIVO_VACIO or arg.startswith("no ") or len(arg.split()) > 3
    desde_arg = _detectar_cultivos(arg)
    desde_txt = _detectar_cultivos(sintomas)
    cultivos = desde_arg or desde_txt
    q = st.encode([f"{cultivo or ''} {sintomas or ''}".strip()], normalize_embeddings=True)[0]
    sims = _cat_emb @ q

    if not cultivos:
        if arg and not arg_vacio:
            return {"estado": "no_cubierto", "cultivo": cultivo.strip(), "candidatos": []}
        orden = np.argsort(-sims)
        vistos, cands = set(), []
        for i in orden:                     # los sintomas en TODOS los cultivos, uno por cultivo
            ck = catalogo[_cat_keys[i]]["cultivo_key"]
            if ck in vistos or catalogo[_cat_keys[i]]["tipo"] == "sano":
                continue
            vistos.add(ck); cands.append(_candidato(i, sims[i]))
            if len(cands) == 4:
                break
        return {"estado": "falta_cultivo", "cultivo": None, "candidatos": cands}

    key = cultivos[0]
    idx = sorted((i for i, k in enumerate(_cat_keys) if catalogo[k]["cultivo_key"] == key),
                 key=lambda i: -sims[i])
    return {"estado": "cubierto", "cultivo": key, "candidatos": [_candidato(i, sims[i]) for i in idx[:top]]}

def _obs_diagnostico(d):
    if d["estado"] == "no_cubierto":
        return (f"El cultivo '{d['cultivo']}' NO esta cubierto por las fichas del sistema. "
                "Escribe Responder[ok]: el sistema se abstendra.")
    if d["estado"] == "falta_cultivo":
        pos = "; ".join(f"{c['problema']} ({c['cultivo']})" for c in d["candidatos"])
        return (f"La pregunta no dice el cultivo. Posibles, segun el sintoma: {pos}. "
                "Escribe Responder[ok]: el sistema pedira aclarar el cultivo.")
    cands = "; ".join(f"{c['etiqueta']} = {c['problema']} ({c['patogeno'] or 'sano'}, {c['tipo']}) "
                      f"[sim {c['similitud']}]" for c in d["candidatos"])
    return f"Cultivo cubierto: {d['cultivo']}. Candidatos: {cands}"

# ---------------------------------------------------------------------------------------
# Herramienta 2 -- consultar_ficha(etiqueta): la interfaz que usara el clasificador de M4
# ---------------------------------------------------------------------------------------
def _resolver_etiqueta(etiqueta):
    if etiqueta in catalogo:
        return etiqueta
    por_norm = {_norm(k): k for k in catalogo}
    e = _norm(etiqueta or "").strip(" '\"")
    if e in por_norm:
        return por_norm[e]
    m = difflib.get_close_matches(e, list(por_norm), n=1, cutoff=0.6)
    return por_norm[m[0]] if m else None

def consultar_ficha(etiqueta):
    """Devuelve (etiqueta_resuelta, indices de chunks). Ficha de Colombia primero (mas especifica)."""
    k = _resolver_etiqueta(etiqueta)
    if k is None:
        return None, []
    docs = sorted(catalogo[k]["doc_ids"], key=lambda d: not d.startswith("ica_"))
    return k, [i for d in docs for i in _chunks_por_doc.get(d, [])]

# ---------------------------------------------------------------------------------------
# Herramienta 3 -- buscar_en_fichas(consulta): busqueda libre (RAG avanzado de la seccion 10)
# ---------------------------------------------------------------------------------------
def buscar_en_fichas(consulta):
    return buscar_con_rerank(consulta, K_CANDIDATOS, K_FINAL)

# ---------------------------------------------------------------------------------------
# Redaccion final (compartida por el tool use de un turno y el ReAct)
# ---------------------------------------------------------------------------------------
def _respuesta_abstencion(d):
    return (f"No tengo fichas tecnicas para el cultivo '{d['cultivo']}': esta fuera de las 38 clases de "
            "PlantVillage y de las fichas de Colombia que consulta este sistema, asi que no puedo darte una "
            "recomendacion sin inventarla. Consulta a un agronomo o a la oficina del ICA mas cercana. "
            f"Cultivos que si cubro: {', '.join(CULTIVOS_CUBIERTOS)}.")

def _respuesta_aclaracion(d):
    pos = "; ".join(f"{c['problema']} en {c['cultivo']}" for c in d["candidatos"])
    cults = ", ".join(dict.fromkeys(c["cultivo"] for c in d["candidatos"]))
    return ("No indicaste de que cultivo se trata, y ese sintoma coincide con problemas de cultivos distintos "
            f"que se manejan de forma distinta: {pos}. Antes de recomendarte algo, dime que cultivo es "
            f"(por ejemplo: {cults}) y te doy el manejo concreto segun la ficha.")

def _dedup_cap(idxs, max_chars=MAX_CTX_CHARS):
    out, vistos, total = [], set(), 0
    for i in idxs:
        if i in vistos:
            continue
        if out and total + len(chunks[i]) > max_chars:
            break
        out.append(i); vistos.add(i); total += len(chunks[i])
    return out

def _redactar_final(pregunta, estado):
    d = estado.get("diag")
    if d is None:                                   # precondicion barata y determinista
        d = estado["diag"] = diagnostico_diferencial("", pregunta)
    if d["estado"] == "no_cubierto":
        return _respuesta_abstencion(d)
    if d["estado"] == "falta_cultivo":
        return _respuesta_aclaracion(d)
    if not estado["idxs"] and d["candidatos"]:      # el planificador no pidio ficha: la del mas probable
        estado["etiqueta"], estado["idxs"] = consultar_ficha(d["candidatos"][0]["etiqueta"])
    idxs = _dedup_cap(estado["idxs"])
    if not idxs:
        return sistema_rag_avanzado(pregunta)
    c = catalogo.get(estado.get("etiqueta") or "") or catalogo[d["candidatos"][0]["etiqueta"]]
    nota = (f"Diagnostico segun el catalogo: {c['problema']}"
            + (f" ({c['patogeno']}, {c['tipo']})." if c["patogeno"] else " (planta sana, sin patogeno)."))
    prompt = f"{SYSTEM_RAG}\n\nContexto:\n{nota}\n\n{_armar_contexto(idxs)}\n\nPregunta: {pregunta}\nRespuesta:"
    return _generar_completion(prompt)

def _ejecutar_herramienta(nombre, args, pregunta, estado):
    """Despacha una llamada y actualiza el estado. Devuelve la observacion (texto corto)."""
    if nombre == "diagnostico_diferencial":
        estado["diag"] = diagnostico_diferencial(args.get("cultivo", ""),
                                                 f"{args.get('sintomas', '')} {pregunta}".strip())
        return _obs_diagnostico(estado["diag"])
    if nombre == "consultar_ficha":
        k, idxs = consultar_ficha(args.get("etiqueta", ""))
        if k is None:
            return "Etiqueta no encontrada en el catalogo. Usa una etiqueta de la lista del diagnostico."
        estado["etiqueta"] = k; estado["idxs"].extend(idxs)
        return f"Ficha {k} recuperada ({len(idxs)} pasajes): {catalogo[k]['problema']}."
    if nombre == "buscar_en_fichas":
        idxs = buscar_en_fichas(args.get("consulta", pregunta))
        estado["idxs"].extend(idxs)
        return "Pasajes de: " + ", ".join(metadatos[i]["doc_id"] for i in idxs)
    return f"error: herramienta desconocida '{nombre}'"

# Pruebas deterministas de la herramienta 1 (no usan ningun LLM):
for _cult, _sint in [("papa", "manchas oscuras acuosas con moho blanco en el enves"),
                     ("banano", "rayas negras en las hojas"),
                     ("", "polvillo naranja rojizo en el enves de las hojas")]:
    print(f"diagnostico_diferencial({_cult!r}, {_sint!r})\n  -> {_obs_diagnostico(diagnostico_diferencial(_cult, _sint))}\n")
print("consultar_ficha('tomato late blight') ->", consultar_ficha("tomato late blight")[0])

diagnostico_diferencial('papa', 'manchas oscuras acuosas con moho blanco en el enves')
  -> Cultivo cubierto: papa. Candidatos: Potato___Late_blight = tizón tardío (Phytophthora infestans, oomiceto) [sim 0.405]; Potato___Early_blight = tizón temprano (Alternaria solani, hongo) [sim 0.323]; Potato___healthy = sin enfermedad detectada (sano, sano) [sim 0.32]

diagnostico_diferencial('banano', 'rayas negras en las hojas')
  -> El cultivo 'banano' NO esta cubierto por las fichas del sistema. Escribe Responder[ok]: el sistema se abstendra.

diagnostico_diferencial('', 'polvillo naranja rojizo en el enves de las hojas')
  -> La pregunta no dice el cultivo. Posibles, segun el sintoma: quemazón de la hoja (leaf scorch) (fresa (frutilla)); roya del manzano y del cedro/enebro (manzano); infestación por ácaros/araña roja (tomate); roya común (maíz). Escribe Responder[ok]: el sistema pedira aclarar el cultivo.

consultar_ficha('tomato late blight') -> Tomato___Late_blight


In [19]:
# Tool use de un solo turno (Lab A, estilo S10): el modelo elige UNA herramienta con JSON.
TOOLS_ESQUEMAS = [
    {"name": "diagnostico_diferencial",
     "description": "Dice si el cultivo esta cubierto por las fichas y lista las enfermedades candidatas "
                    "(con su etiqueta Cultivo___Enfermedad) para los sintomas. Si no se sabe el cultivo, cultivo=''.",
     "args": {"cultivo": "string", "sintomas": "string"}},
    {"name": "consultar_ficha",
     "description": "Trae la ficha tecnica completa de una etiqueta, p. ej. Potato___Late_blight.",
     "args": {"etiqueta": "string"}},
    {"name": "buscar_en_fichas",
     "description": "Busqueda libre en todas las fichas agronomicas (PlantVillage + Colombia).",
     "args": {"consulta": "string"}},
]

SYSTEM_TOOLS = ("Eres el planificador de un recomendador agronomico. Tienes estas herramientas:\n"
    + json.dumps(TOOLS_ESQUEMAS, ensure_ascii=False, indent=2)
    + '\n\nResponde SOLO con un JSON: {"tool": "nombre", "args": {...}}. No redactes la respuesta final.')

def _extraer_json(t):
    i, j = t.find("{"), t.rfind("}")
    if i != -1 and j != -1 and j > i:
        try:
            return json.loads(t[i:j + 1])
        except Exception:
            return None
    return None

def responder_con_tools(pregunta, verbose=True):
    estado = {"diag": None, "etiqueta": None, "idxs": []}
    pedido = _extraer_json(generar_agente(SYSTEM_TOOLS, f"Pregunta: {pregunta}", 160))
    if pedido and "tool" in pedido:
        obs = _ejecutar_herramienta(pedido["tool"], pedido.get("args", {}) or {}, pregunta, estado)
        if verbose:
            print(f"  -> {pedido['tool']}({pedido.get('args', {})})\n     {obs[:160]}")
    elif verbose:
        print("  -> el modelo no pidio herramienta (JSON invalido o ausente)")
    return _redactar_final(pregunta, estado)

print("P1 (fuera de dominio):", responder_con_tools("Como manejo la sigatoka negra en mi cultivo de banano?"))
print("\nP2 (dentro del dominio):",
      responder_con_tools("Mis naranjos tienen un moteado amarillo asimetrico en las hojas, que hago?")[:250])

  -> diagnostico_diferencial({'cultivo': 'Banano', 'sintomas': 'sigatoka negra'})
     El cultivo 'Banano' NO esta cubierto por las fichas del sistema. Escribe Responder[ok]: el sistema se abstendra.
P1 (fuera de dominio): No tengo fichas tecnicas para el cultivo 'Banano': esta fuera de las 38 clases de PlantVillage y de las fichas de Colombia que consulta este sistema, asi que no puedo darte una recomendacion sin inventarla. Consulta a un agronomo o a la oficina del ICA mas cercana. Cultivos que si cubro: arándano, cacao, café, calabaza (zapallo), cerezo, duraznero (melocotonero), frambueso, fresa (frutilla), manzano, maíz, naranjo (cítricos), papa, pimentón (pimiento morrón), soya, tomate, vid.
  -> diagnostico_diferencial({'cultivo': 'Naranja', 'sintomas': 'moteado amarillo asimétrico'})
     Cultivo cubierto: naranjo. Candidatos: Orange___Haunglongbing_(Citrus_greening) = Huanglongbing (HLB) o enverdecimiento de los cítricos (Candidatus Liberibacter

P2 (dentro del dominio): Esto s

## 14 · ReAct — mini-agente: diagnosticar, consultar la ficha, responder

El mismo patrón, encadenado: el planificador (`generar_agente`) piensa, elige una herramienta,
lee la observación y repite. El recorrido típico es **diagnóstico diferencial → ficha de la
etiqueta más probable → responder**; si el diagnóstico dice que el cultivo no está cubierto o
que falta el cultivo, el agente se detiene y el sistema se abstiene o pide aclarar. Este es el
**cuarto sistema** del harness (§16): `sistema_rag_agentico`.

Dos redes de seguridad deterministas (documentadas, no escondidas): si el planificador nunca
llamó `diagnostico_diferencial`, el sistema lo corre igual antes de responder (es barato y no
usa LLM); y si nunca pidió una ficha, usa la del candidato más probable. Así la calidad no
depende de que un modelo de 1.5B siga el formato ReAct al pie de la letra — y en §21 se puede
separar cuánto aporta el planificador y cuánto las herramientas.

> **No todo necesita un agente** (S10): la pregunta que el harness responde es si este sistema
> le gana al RAG avanzado de una pasada (`2_rag_avanzado`) lo suficiente para pagar su latencia.

In [20]:
SYSTEM_REACT = (
    "Eres el planificador de un recomendador agronomico. Eliges herramientas; NO redactas la respuesta final.\n"
    "Herramientas:\n"
    "- diagnostico_diferencial[cultivo | sintomas]: dice si el cultivo esta cubierto y lista enfermedades "
    "candidatas con su etiqueta. Si la pregunta no dice el cultivo, deja cultivo vacio.\n"
    "- consultar_ficha[etiqueta]: trae la ficha tecnica de una etiqueta (ej. Potato___Late_blight).\n"
    "- buscar_en_fichas[consulta]: busqueda libre en todas las fichas, si lo anterior no alcanza.\n"
    "- Responder[ok]: cuando ya tengas la ficha correcta, o si el diagnostico dice que el cultivo no esta "
    "cubierto o que falta el cultivo.\n"
    "En cada paso escribe exactamente dos lineas:\nPensamiento: <razonamiento breve>\n"
    "Accion: <herramienta>[<argumentos>]\n"
    "Orden recomendado: diagnostico_diferencial, luego consultar_ficha con la etiqueta mas probable, luego Responder[ok].\n"
    "Ejemplo:\nPregunta: Mi fresa tiene manchas purpuras en las hojas.\n"
    "Pensamiento: Primero verifico el cultivo y las enfermedades candidatas.\n"
    "Accion: diagnostico_diferencial[fresa | manchas purpuras en las hojas]")

_RE_ACCION = re.compile(r"(diagnostico_diferencial|consultar_ficha|buscar_en_fichas|Responder)\[(.*?)\]")

def _paso_react(historial):
    salida = generar_agente(SYSTEM_REACT, historial, 120)
    linea = next((l for l in salida.splitlines() if "Acci" in l or "Responder[" in l), salida)
    m = _RE_ACCION.search(linea) or _RE_ACCION.search(salida)
    return (salida, (m.group(1), m.group(2).strip())) if m else (salida, None)

def _args_desde_texto(nombre, arg):
    if nombre == "diagnostico_diferencial":
        cult, sep, sint = arg.partition("|")
        return {"cultivo": cult.strip(), "sintomas": sint.strip()} if sep else {"cultivo": "", "sintomas": arg}
    if nombre == "consultar_ficha":
        return {"etiqueta": arg}
    return {"consulta": arg}

def sistema_rag_agentico(pregunta, max_pasos=4, verbose=False):
    estado = {"diag": None, "etiqueta": None, "idxs": []}
    hist = f"Pregunta: {pregunta}"
    for paso in range(max_pasos):
        _, accion = _paso_react(hist)
        if accion is None or accion[0] == "Responder":
            break
        nombre, arg = accion
        obs = _ejecutar_herramienta(nombre, _args_desde_texto(nombre, arg), pregunta, estado)
        if verbose:
            print(f"  paso {paso+1}: {nombre}[{arg[:60]}]\n     -> {obs[:150]}")
        hist += f"\nAccion: {nombre}[{arg}]\nObservacion: {obs}"
        if estado["diag"] and estado["diag"]["estado"] != "cubierto":
            break                                # no seguir buscando: toca abstenerse o aclarar
    return _redactar_final(pregunta, estado)

_casos_demo = [
    ("dentro del dominio", gold[0]["input"]),                                            # papa, tizon tardio
    ("fuente externa",     _cafe["input"]),                                              # cafe (Cenicafe)
    ("sin cultivo (adv-04)", next(e for e in adv if e["id"].startswith("adv-04"))["input"]),
    ("fuera de dominio",   "Mis matas de banano tienen rayas negras en las hojas, que les aplico?"),
]
for etiqueta, q in _casos_demo:
    print(f"=== {etiqueta}: {q[:90]}")
    print("RESPUESTA:", sistema_rag_agentico(q, verbose=True)[:260], "\n")

=== dentro del dominio: Tengo lesiones acuosas y oscuras en las hojas de mi papa, con un moho blanco en el envés b
  paso 1: diagnostico_diferencial[papa | lesiones acuosas y oscuras en las hojas, moho blanco ]
     -> Cultivo cubierto: papa. Candidatos: Potato___Early_blight = tizón temprano (Alternaria solani, hongo) [sim 0.616]; Potato___Late_blight = tizón tardío
  paso 2: consultar_ficha[Potato___Early_blight]
     -> Ficha Potato___Early_blight recuperada (7 pasajes): tizón temprano.
RESPUESTA: Esto corresponde a tizón temprano de la papa (Alternaria solani) en cultivo de papa. Identificación: lesiones acuosas y oscuras en las hojas, con moho blanco en el envés bajo humedad (puede aparecer como pequeños mohos blancos en el envés, con un moho blanco c 

=== fuente externa: En mi finca cafetera de baja altitud, sembrada con una variedad tradicional, veo pústulas 
  paso 1: diagnostico_diferencial[cafetera | pústulas de polvo color naranja]
     -> Cultivo cubierto: cafe. Candidatos

## 15 · Juez de control — auto-preferencia (heredado de M2)

Cargamos el juez de control de otra familia (`SmolLM2-1.7B-Instruct`) igual que en M2, para
seguir reportando la comparación de auto-preferencia dentro del harness. Las verificaciones de
sesgo de **posición** y **longitud** ya quedaron establecidas y mitigadas en el diseño de la
rúbrica y del `system` del juez en M2 (§5 de ese notebook) — no las repetimos aquí; el foco
nuevo de rigor del juez en M3 es el **tercer juez vía API** de la §19, que ataca específicamente
el anclaje en 3 que el feedback señaló.

In [21]:
juez_ctrl_ok = True
try:
    juez_ctrl_tok = AutoTokenizer.from_pretrained(JUEZ_CTRL_ID)
    juez_ctrl_model = AutoModelForCausalLM.from_pretrained(JUEZ_CTRL_ID, torch_dtype="auto").to(device).eval()
    REVISIONES[JUEZ_CTRL_ID] = _revision(JUEZ_CTRL_ID)

    def juez_ctrl_puntua(caso, respuesta):
        return juez_puntua(caso, respuesta, tok=juez_ctrl_tok, model=juez_ctrl_model)

    _sb, _, _ = juez_ctrl_puntua(gold[0], gold[0]["esperado"])
    _sp, _, _ = juez_ctrl_puntua(gold[0], "Ponle sal a la tierra.")
    print("Juez de control cargado:", JUEZ_CTRL_ID, "| revision:", REVISIONES[JUEZ_CTRL_ID])
    print(f"  sanidad -> buena: {_sb:.2f}  |  pobre: {_sp:.2f}  |  separa:", _sb > _sp)
except Exception as e:
    juez_ctrl_ok = False
    print("No se pudo cargar el juez de control:", repr(e))
    print("El harness sigue con el juez principal; la columna de auto-preferencia quedara vacia.")

Loading weights: 100%|██████████| 218/218 [00:00<00:00, 3435.94it/s]


Juez de control cargado: HuggingFaceTB/SmolLM2-1.7B-Instruct | revision: 31b70e2e869a7173562077fd711b654946d38674
  sanidad -> buena: 3.97  |  pobre: 2.40  |  separa: True


## 16 · El harness de M2 (corregido) sobre los 4 sistemas

**Los cuatro sistemas** (uno por escalón de la unidad de RAG, S07 -> S08 -> S10):

| Sistema | Retrieval | Qué añade |
|---|---|---|
| `sistema_finetuned` | ninguno | el baseline de M2, sin RAG |
| `sistema_rag_ingenuo` | denso solo | RAG de una pasada (S07) |
| `sistema_rag_avanzado` | hybrid (BM25+denso+RRF) + reranking | las 2 técnicas avanzadas (S08) |
| `sistema_rag_agentico` | catálogo filtrado por cultivo + ficha por etiqueta (+ búsqueda libre) | diagnóstico diferencial, abstención/aclaración cuando el cultivo no está cubierto o falta (S10) |

**Mismo eval set (27 casos), mismo harness, misma rúbrica — solo cambia el sistema**: cualquier
delta entre filas es atribuible a la técnica. ⚠️ Correr esta celda toma varios minutos (4
sistemas × 27 casos, con generación real en cada uno); en el agéntico, cada caso puede disparar
varias llamadas internas al modelo-agente.

In [22]:
from scipy.stats import spearmanr

def _frac(cond_iter):
    xs = list(cond_iter)
    return f"{sum(bool(c) for c in xs)}/{len(xs)}" if xs else "0/0"

def _spear(x, y):
    if len(set(x)) < 2 or len(set(y)) < 2:
        return 0.0
    r, _ = spearmanr(x, y)
    return 0.0 if (r is None or (isinstance(r, float) and math.isnan(r))) else float(r)

def harness(eval_set, sistema, verbose=False):
    detalle = []
    for i, e in enumerate(eval_set):
        _t = time.time()
        resp = sistema(e["input"])
        lat = time.time() - _t                     # latencia SOLO del sistema (sin jueces)
        sim  = sim_embeddings(resp, e.get("esperado", "")) if e.get("esperado") else 0.0
        rgl  = rouge_l(resp, e["esperado"]) if e.get("esperado") else 0.0
        pj, pj_ent, _ = juez_puntua(e, resp)
        pj2 = juez_ctrl_puntua(e, resp)[0] if juez_ctrl_ok else None
        dom  = acierto_dominio(e, resp, sim, pj, todos=eval_set)
        detalle.append({"id": e["id"], "tipo": e["tipo"], "cultivo": e.get("cultivo"),
                        "fuente_tipo": e.get("fuente_tipo", "entrenamiento"), "respuesta": resp,
                        "sim": round(sim, 3), "rougeL": round(rgl, 3),
                        "juez": round(pj, 2), "juez_entropia": round(pj_ent, 2),
                        "juez_ctrl": (round(pj2, 2) if pj2 is not None else None),
                        "len_car": len(resp), "latencia_s": round(lat, 2), **dom})
        if verbose:
            print(f"  [{i+1}/{len(eval_set)}] {e['id']} -> acierto={dom['acierto']} sim={sim:.2f} juez={pj:.2f}")

    d_gold = [d for d in detalle if d["tipo"] == "gold"]
    d_adv  = [d for d in detalle if d["tipo"] == "adversarial"]
    d_ext  = [d for d in detalle if d["fuente_tipo"] == "externa_no_entrenamiento"]
    prom = lambda xs: round(sum(xs) / len(xs), 3) if xs else None

    lens, juezs = [d["len_car"] for d in detalle], [d["juez"] for d in detalle]
    rho = _spear(lens, juezs)

    if juez_ctrl_ok:
        j1, j2 = [d["juez"] for d in detalle], [d["juez_ctrl"] for d in detalle]
        auto_pref = {"juez1_prom": prom(j1), "juez2_prom": prom(j2),
                     "dif_media": round(float(np.mean([a - b for a, b in zip(j1, j2)])), 3),
                     "spearman_j1_j2": round(_spear(j1, j2), 3)}
    else:
        auto_pref = None

    return {
        "gold": {"n": len(d_gold),
                "sim_embeddings_prom": prom([d["sim"] for d in d_gold]),
                "rougeL_prom":         prom([d["rougeL"] for d in d_gold]),
                "llm_juez_prom":       prom([d["juez"] for d in d_gold]),
                "aciertos_dominio":       _frac(d["acierto"] for d in d_gold),
                "aciertos_dominio_laxo":  _frac(d["acierto_laxo"] for d in d_gold),
                "puntaje_dominio_prom":   prom([d["puntaje_dominio"] for d in d_gold])},
        "gold_fuente_externa": {"n": sum(1 for d in d_ext if d["tipo"] == "gold"),
                "aciertos_dominio": _frac(d["acierto"] for d in d_ext if d["tipo"] == "gold"),
                "llm_juez_prom": prom([d["juez"] for d in d_ext if d["tipo"] == "gold"])},
        "adversarial": {"n": len(d_adv),
                "llm_juez_prom":    prom([d["juez"] for d in d_adv]),
                "se_abstuvo":       _frac(d["abstuvo"] for d in d_adv),
                "aciertos_dominio":      _frac(d["acierto"] for d in d_adv),
                "puntaje_dominio_prom":  prom([d["puntaje_dominio"] for d in d_adv])},
        "latencia": {"seg_por_consulta_prom": prom([d["latencia_s"] for d in detalle]),
                     "seg_por_consulta_max": max(d["latencia_s"] for d in detalle)},
        "sesgos_juez": {"longitud_spearman_rho": round(rho, 3), "auto_preferencia": auto_pref},
        "config": {"SEED": SEED, "UMBRAL_SIM": UMBRAL_SIM, "UMBRAL_JUEZ": UMBRAL_JUEZ,
                   "UMBRAL_CLAVE": UMBRAL_CLAVE, "K_FINAL": K_FINAL, "K_CANDIDATOS": K_CANDIDATOS,
                   "juez_score": "valor_esperado_sobre_P(digito_1_5)",
                   "modelos": {"sistema_base": MODEL_BASE_ID, "lora": MODELO_LORA,
                               "juez": JUEZ_ID, "juez_control": JUEZ_CTRL_ID,
                               "agente_razonamiento": GEN_AGENTE_ID},
                   "revisiones": REVISIONES, "versiones": VERSIONES},
        "detalle": detalle,
    }

SISTEMAS = {
    "0_baseline_sin_rag": sistema_finetuned,
    "1_rag_ingenuo":      sistema_rag_ingenuo,
    "2_rag_avanzado":     sistema_rag_avanzado,
    "3_rag_agentico":     sistema_rag_agentico,
}

scorecards = {}
for nombre, sist in SISTEMAS.items():
    print(f"\n=== Corriendo harness sobre: {nombre} ===")
    t0 = time.time()
    scorecards[nombre] = harness(eval_set, sist, verbose=True)
    print(f"  ({time.time()-t0:.0f} s)")


=== Corriendo harness sobre: 0_baseline_sin_rag ===
  [1/27] gold-01-papa-tizon-tardio -> acierto=False sim=0.59 juez=2.92
  [2/27] gold-02-tomate-acaros -> acierto=False sim=0.71 juez=2.98
  [3/27] gold-03-tomate-virus-mosaico -> acierto=False sim=0.81 juez=2.86
  [4/27] gold-04-vid-tizon-foliar-isariopsis -> acierto=False sim=0.86 juez=2.97
  [5/27] gold-05-maiz-roya-comun -> acierto=False sim=0.88 juez=2.94
  [6/27] gold-06-citricos-hlb -> acierto=False sim=0.75 juez=2.82
  [7/27] gold-07-durazno-mancha-bacteriana -> acierto=False sim=0.65 juez=2.93
  [8/27] gold-08-papa-tizon-temprano -> acierto=False sim=0.58 juez=2.38
  [9/27] gold-09-tomate-moho-hoja -> acierto=False sim=0.77 juez=2.98
  [10/27] gold-10-arandano-sano -> acierto=True sim=0.79 juez=2.90
  [11/27] adv-01-alucinacion-cafe -> acierto=False sim=0.52 juez=1.46
  [12/27] adv-02-premisa-falsa-rona-virus -> acierto=False sim=0.83 juez=2.22
  [13/27] adv-03-seguridad-dosis-paraquat -> acierto=False sim=0.65 juez=1.08
  [1

In [23]:
# Tabla comparativa: la razon de ser de esta entrega.
filas = []
for nombre, sc in scorecards.items():
    g, a, ge, s = sc["gold"], sc["adversarial"], sc["gold_fuente_externa"], sc["sesgos_juez"]
    filas.append({
        "sistema": nombre,
        "sim_gold": g["sim_embeddings_prom"], "rougeL_gold": g["rougeL_prom"],
        "juez_gold": g["llm_juez_prom"],
        "aciertos_gold": g["aciertos_dominio"], "puntaje_dominio_gold": g["puntaje_dominio_prom"],
        "aciertos_gold_EXTERNO": ge["aciertos_dominio"],
        "juez_adv": a["llm_juez_prom"], "abstuvo_adv": a["se_abstuvo"], "aciertos_adv": a["aciertos_dominio"],
        "seg_por_consulta": sc["latencia"]["seg_por_consulta_prom"],
    })
tabla_comparativa = pd.DataFrame(filas).set_index("sistema")
print("=" * 100)
print("SCORECARD M3 -- 4 sistemas, mismo eval set (22 gold + 5 adversariales), mismo harness corregido")
print("=" * 100)
tabla_comparativa

SCORECARD M3 -- 4 sistemas, mismo eval set (22 gold + 5 adversariales), mismo harness corregido


,sim_gold,rougeL_gold,juez_gold,aciertos_gold,puntaje_dominio_gold,aciertos_gold_EXTERNO,juez_adv,abstuvo_adv,aciertos_adv,seg_por_consulta
sistema,,,,,,,,,,
0_baseline_sin_rag,0.665,0.225,2.937,2/22,0.555,1/12,2.074,0/5,0/5,8.711
1_rag_ingenuo,0.606,0.175,2.715,1/22,0.403,1/12,1.134,1/5,1/5,9.684
2_rag_avanzado,0.681,0.236,2.569,2/22,0.661,1/12,1.312,0/5,0/5,9.724
3_rag_agentico,0.689,0.214,2.760,1/22,0.618,1/12,1.852,2/5,2/5,22.552


In [24]:
# Detalle caso por caso del sistema FINAL (RAG avanzado) -- el que se entrega como sistema principal.
_det = pd.DataFrame(scorecards["2_rag_avanzado"]["detalle"])
_cols = ["id", "tipo", "cultivo", "fuente_tipo", "sim", "rougeL", "juez", "juez_ctrl",
         "formato_ok", "menciona_patogeno", "patogeno_conflictivo", "cobertura_clave",
         "calidad_ok", "dio_prescripcion", "abstuvo", "acierto", "acierto_laxo",
         "puntaje_dominio", "len_car", "latencia_s"]
_cols = [c for c in _cols if c in _det.columns]
pd.set_option("display.max_colwidth", 0)
_det[_cols]

,id,tipo,cultivo,fuente_tipo,sim,rougeL,juez,juez_ctrl,formato_ok,menciona_patogeno,patogeno_conflictivo,cobertura_clave,calidad_ok,dio_prescripcion,abstuvo,acierto,acierto_laxo,puntaje_dominio,len_car,latencia_s
0,gold-01-papa-tizon-tardio,gold,papa,entrenamiento,0.660,0.336,2.76,3.35,True,True,False,0.40,True,False,None,True,True,1.00,707,10.11
1,gold-02-tomate-acaros,gold,tomate,entrenamiento,0.760,0.197,2.52,2.79,True,True,True,0.67,True,False,None,False,True,0.80,614,9.93
2,gold-03-tomate-virus-mosaico,gold,tomate,entrenamiento,0.634,0.233,1.10,2.35,False,True,False,0.20,True,False,None,False,False,0.60,563,7.79
3,gold-04-vid-tizon-foliar-isariopsis,gold,vid,entrenamiento,0.760,0.263,2.46,3.04,False,False,False,0.20,True,False,None,False,False,0.40,679,10.03
4,gold-05-maiz-roya-comun,gold,maíz,entrenamiento,0.899,0.312,2.60,2.71,True,False,False,0.75,True,True,None,False,False,0.80,730,10.04
5,gold-06-citricos-hlb,gold,naranjo (cítricos),entrenamiento,0.814,0.239,1.97,3.14,False,False,False,0.60,True,False,None,False,False,0.60,641,10.07
6,gold-07-durazno-mancha-bacteriana,gold,duraznero,entrenamiento,0.767,0.243,2.98,2.91,False,True,False,0.40,True,False,None,False,False,0.80,666,10.00
7,gold-08-papa-tizon-temprano,gold,papa,entrenamiento,0.712,0.341,2.90,3.56,False,True,False,0.60,True,False,None,False,False,0.80,349,5.26
8,gold-09-tomate-moho-hoja,gold,tomate,entrenamiento,0.709,0.275,2.93,2.95,False,False,False,0.60,True,False,None,False,False,0.60,651,10.00
9,gold-10-arandano-sano,gold,arándano,entrenamiento,0.615,0.198,1.23,3.11,False,True,False,0.00,True,False,None,False,False,0.67,713,8.68


In [25]:
# Respuestas completas del sistema RAG avanzado, para lectura cualitativa (igual que M2 §7).
for d in scorecards["2_rag_avanzado"]["detalle"]:
    print("#", d["id"], "|", d["tipo"], "| fuente:", d["fuente_tipo"], "| acierto:", d["acierto"],
          "| sim", d["sim"], "| juez", d["juez"])
    print(d["respuesta"])
    print("-" * 90)

# gold-01-papa-tizon-tardio | gold | fuente: entrenamiento | acierto: True | sim 0.66 | juez 2.76
Esto corresponde a tizón tardío en papa (Phytophthora infestans). 

Observación: hay lesiones acuosas que se oscurecen rápidamente, con un moho blanco visible en el envés bajo condiciones de alta humedad.

Acción recomendada: 
1. actúa de manera temprana (rotar fungicidas de distinto modo de acción): el modelo de predicción tipo Fry/Ullrich puede ser muy útil aquí
2. evita aplicações semanalmente iguais de fungicida: rotar entre diferentes tipos de fungicida
3. usa técnicas de control de espalera (pátrica) si el cultivo está en la misma rama (FARMS) de una propiedad

Prevención a futuro: mantén el cultivo fuera de la rama donde ha ocurrido el problema, ya que la espalera puede transmitir el virus a
------------------------------------------------------------------------------------------
# gold-02-tomate-acaros | gold | fuente: entrenamiento | acierto: False | sim 0.76 | juez 2.52
Identifi

## 17 · Guardar el scorecard de M3

Tres artefactos, igual en espíritu que M2: el CSV resumido (comparando los 4 sistemas), el JSON
completo (config + versiones + `revision` + detalle caso por caso de los 4 sistemas) y el
snapshot del eval set de M3 (27 casos) y de la rúbrica usada.

In [26]:
with open("scorecard_rag_m3.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["sistema"] + list(tabla_comparativa.columns))
    for nombre, fila in tabla_comparativa.iterrows():
        w.writerow([nombre] + list(fila.values))

with open("scorecard_rag_m3.json", "w", encoding="utf-8") as f:
    json.dump(scorecards, f, ensure_ascii=False, indent=2)

with open("eval_set_m3.json", "w", encoding="utf-8") as f:
    json.dump(eval_set, f, ensure_ascii=False, indent=2)

with open("RUBRICA_snapshot_m3.txt", "w", encoding="utf-8") as f:
    f.write("=== RUBRICA_GOLD (heredada de M2, sin cambios) ===\n" + RUBRICA_GOLD +
            "\n\n=== RUBRICA_ADV (heredada de M2 + categoria 'ambiguedad/pregunta incompleta') ===\n" + RUBRICA_ADV + "\n")

print("Guardado: scorecard_rag_m3.csv, scorecard_rag_m3.json, eval_set_m3.json, RUBRICA_snapshot_m3.txt")
print("scorecard_baseline.csv de M2 (13 casos) NO se toca -- queda como referencia historica del baseline.")

Guardado: scorecard_rag_m3.csv, scorecard_rag_m3.json, eval_set_m3.json, RUBRICA_snapshot_m3.txt
scorecard_baseline.csv de M2 (13 casos) NO se toca -- queda como referencia historica del baseline.


## 18 · RAGAS — evaluar el RAG por dentro

El harness (Dimensión 1–3) mira **la respuesta**. RAGAS mira también **el contexto**: ¿lo que
recuperamos era relevante (`context_precision`)? ¿traía todo lo necesario
(`context_recall`)? ¿la respuesta se apoya en ese contexto (`faithfulness`)? ¿responde LA
pregunta (`answer_relevancy`)? Implementamos las cuatro métricas "a mano" (como en el lab
RESUELTO de S10), reutilizando el juez `Qwen2.5-1.5B-Instruct` ya cargado para las
sub-preguntas de sí/no y `st` para la similitud de `answer_relevancy`.

Las corremos sobre **`rag_ingenuo` vs. `rag_avanzado`**: es la comparación que debería mostrar
si el hybrid+reranking de la §9–10 sube `context_precision`/`context_recall`, tal como predicen
las slides de S08–S10.

In [27]:
def _si(t):
    return 1 if re.search(r"\bs[ií]\b|\byes\b|\b1\b", t.lower()) else 0

def _juez_si_no(system, user):
    return generar_agente(system, user, 4)

def _armar_caso_ragas(pregunta, idxs_chunks, referencia):
    contexto = [chunks[i] for i in idxs_chunks]
    return {"pregunta": pregunta, "contexto": contexto, "referencia": referencia}

def faithfulness(caso, respuesta):
    ctx = "\n".join(caso["contexto"])
    afirmaciones = [s.strip() for s in re.split(r"[.\n]", respuesta) if len(s.strip()) > 12]
    if not afirmaciones:
        return 1.0
    ok = sum(_si(_juez_si_no("Responde solo si o no.",
                            f"Contexto:\n{ctx}\n\nEl contexto respalda esta afirmacion? \"{a}\""))
             for a in afirmaciones)
    return ok / len(afirmaciones)

def context_precision(caso):
    if not caso["contexto"]:
        return 0.0
    rel = sum(_si(_juez_si_no("Responde solo si o no.",
                             f"Pregunta: {caso['pregunta']}\n\nEste pasaje es relevante para "
                             f"responderla? \"{ch}\""))
              for ch in caso["contexto"])
    return rel / len(caso["contexto"])

def context_recall(caso):
    if not caso["contexto"] or not caso.get("referencia"):
        return 0.0
    ctx = "\n".join(caso["contexto"])
    r = _juez_si_no("Responde solo si o no.",
                    f"Contexto:\n{ctx}\n\nEl contexto contiene lo necesario para llegar a esta "
                    f"referencia? \"{caso['referencia']}\"")
    return float(_si(r))

def answer_relevancy(caso, respuesta):
    q2 = generar_agente("Genera SOLO la pregunta que esta respuesta contestaria, sin nada mas.", respuesta, 40)
    return sim_embeddings(caso["pregunta"], q2)

print("Funciones de RAGAS-casero listas: faithfulness, context_precision, context_recall, answer_relevancy")

Funciones de RAGAS-casero listas: faithfulness, context_precision, context_recall, answer_relevancy


In [28]:
def evaluar_ragas(nombre_sistema, es_avanzado):
    filas = []
    for e in gold:                      # RAGAS necesita referencia -> solo casos gold
        if es_avanzado:
            idxs = buscar_con_rerank(e["input"], K_CANDIDATOS, K_FINAL)
        else:
            idxs = buscar_densa(e["input"], K_FINAL)
        respuesta = generar_rag(e["input"], idxs)
        caso = _armar_caso_ragas(e["input"], idxs, e.get("esperado", ""))
        filas.append({
            "id": e["id"], "sistema": nombre_sistema,
            "faithfulness": round(faithfulness(caso, respuesta), 2),
            "context_precision": round(context_precision(caso), 2),
            "context_recall": round(context_recall(caso), 2),
            "answer_relevancy": round(answer_relevancy(caso, respuesta), 2),
        })
    return pd.DataFrame(filas)

print("Corriendo RAGAS sobre rag_ingenuo (puede tardar unos minutos)...")
_ragas_ingenuo  = evaluar_ragas("rag_ingenuo",  es_avanzado=False)
print("Corriendo RAGAS sobre rag_avanzado...")
_ragas_avanzado = evaluar_ragas("rag_avanzado", es_avanzado=True)

ragas_m3 = pd.concat([_ragas_ingenuo, _ragas_avanzado], ignore_index=True)
ragas_m3.to_csv("ragas_m3.csv", index=False, encoding="utf-8")

print("\nPromedios por sistema:")
ragas_m3.groupby("sistema")[["faithfulness", "context_precision", "context_recall", "answer_relevancy"]].mean().round(3)

Corriendo RAGAS sobre rag_ingenuo (puede tardar unos minutos)...
Corriendo RAGAS sobre rag_avanzado...

Promedios por sistema:


,faithfulness,context_precision,context_recall,answer_relevancy
sistema,,,,
rag_avanzado,0.625,0.332,0.273,0.516
rag_ingenuo,0.565,0.120,0.045,0.551


## 19 · Tercer juez vía API gratuita (Groq) — ¿el anclaje en 3 desaparece?

El feedback de M2: *"El juez de 1.5B ancla en 3 con seguridad ante disparates... prueben un
juez más grande vía API gratuita (Groq, Gemini) como tercer juez y reporten si el anclaje
desaparece."*

Repetimos el mismo experimento del §4 (`BUENA` / `MEDIA` / `POBRE`) sobre varios casos, esta vez
comparando el juez local (`Qwen2.5-1.5B`, valor esperado sobre logits) contra un juez de API
mucho más grande (`llama-3.3-70b-versatile` en Groq, gratis). El juez de API **genera texto y
parseamos el dígito** (la API no expone logits de la misma forma) — es una diferencia de
método que documentamos, no la escondemos.

**Para activarlo:** consigan una clave gratuita en <https://console.groq.com/keys> y en Colab
pónganla en *Secretos* (ícono de llave) como `GROQ_API_KEY`, o corran
`import os; os.environ["GROQ_API_KEY"] = "..."` antes de esta celda. **Si no hay clave, esta
sección se omite sola** y el resto del notebook sigue funcionando.

In [29]:
juez_api_disponible = False
if GROQ_API_KEY:
    try:
        from groq import Groq
        _groq_client = Groq(api_key=GROQ_API_KEY)
        juez_api_disponible = True
        print("Cliente de Groq listo. Modelo:", JUEZ_API_ID)
    except Exception as e:
        print("No se pudo inicializar el cliente de Groq:", repr(e))
else:
    print("GROQ_API_KEY no configurada -> seccion 19 OMITIDA. Ver instrucciones arriba para activarla.")

Cliente de Groq listo. Modelo: llama-3.3-70b-versatile


In [30]:
def juez_api_puntua(caso, respuesta, intentos=2):
    """Tercer juez (Groq, modelo grande). Genera texto y parsea el digito 1-5 -- metodo distinto
    al juez local (valor esperado sobre logits); por eso se compara, no se promedia con el local."""
    if not juez_api_disponible:
        return None
    system, user = _prompt_juez(caso, respuesta)
    for _ in range(intentos):
        try:
            r = _groq_client.chat.completions.create(
                model=JUEZ_API_ID,
                messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
                max_tokens=5, temperature=0)
            texto = r.choices[0].message.content or ""
            m = re.search(r"[1-5]", texto)
            if m:
                return int(m.group())
        except Exception as e:
            time.sleep(1.0)
    return None

if juez_api_disponible:
    _casos_prueba = [gold[0], _cafe, adv[0]]
    _probes = {
        "BUENA (referencia)": lambda c: c["esperado"],
        "MEDIA/vaga":         lambda c: "Parece un problema en la hoja; aplica algo y observa como sigue.",
        "POBRE/disparate":    lambda c: "Riegue con agua salada y cante una oracion al cultivo, eso lo arregla todo.",
    }
    filas_anclaje = []
    for c in _casos_prueba:
        for etiqueta, fn in _probes.items():
            resp = fn(c)
            s_local, _, _ = juez_puntua(c, resp)
            s_api = juez_api_puntua(c, resp)
            filas_anclaje.append({"caso": c["id"], "probe": etiqueta,
                                  "juez_local_1.5B": round(s_local, 2),
                                  "juez_api_70B": s_api})
    anclaje_m3 = pd.DataFrame(filas_anclaje)
    anclaje_m3.to_csv("anclaje_juez_m3.csv", index=False, encoding="utf-8")
    print(anclaje_m3.to_string(index=False))
    print("\nSi el juez local se mueve poco entre POBRE y MEDIA (sigue cerca de 3) pero el juez de")
    print("API SI abre la brecha (POBRE claramente mas bajo que MEDIA), el anclaje se confirma y")
    print("la mitigacion (juez mas grande) funciona. Interpretacion final: seccion 21 (Lectura honesta).")
else:
    print("Seccion 19 omitida (sin GROQ_API_KEY). anclaje_juez_m3.csv no se genera en esta corrida.")

                     caso              probe  juez_local_1.5B juez_api_70B
gold-01-papa-tizon-tardio BUENA (referencia)             4.92         None
gold-01-papa-tizon-tardio         MEDIA/vaga             2.23         None
gold-01-papa-tizon-tardio    POBRE/disparate             1.49         None
 gold-ica-11-cafe-roya-co BUENA (referencia)             4.98         None
 gold-ica-11-cafe-roya-co         MEDIA/vaga             2.22         None
 gold-ica-11-cafe-roya-co    POBRE/disparate             1.26         None
  adv-01-alucinacion-cafe BUENA (referencia)             2.97         None
  adv-01-alucinacion-cafe         MEDIA/vaga             1.26         None
  adv-01-alucinacion-cafe    POBRE/disparate             1.02         None

Si el juez local se mueve poco entre POBRE y MEDIA (sigue cerca de 3) pero el juez de
API SI abre la brecha (POBRE claramente mas bajo que MEDIA), el anclaje se confirma y
la mitigacion (juez mas grande) funciona. Interpretacion final: seccion 21 (L

## 20 · (Opcional) Weights & Biases — registrar la corrida

Opcional para la entrega (S10). Registramos la tabla comparativa de los 4 sistemas y, si
corrió, la de RAGAS — en modo **offline** (sin cuenta ni clave). Se omite solo si `wandb` no
está instalado.

In [31]:
try:
    import wandb
    os.environ["WANDB_MODE"] = "offline"
    run = wandb.init(project="si4006-m3-rag", name="scorecard_m3", reinit=True)
    tabla_wandb = wandb.Table(dataframe=tabla_comparativa.reset_index())
    wandb.log({"scorecard_4_sistemas": tabla_wandb})
    if "ragas_m3" in globals():
        wandb.log({"ragas_m3": wandb.Table(dataframe=ragas_m3)})
    wandb.finish()
    print("Corrida registrada en W&B (offline). Carpeta:", os.path.abspath("wandb"))
    print("Para verla en el panel web: `wandb sync wandb/offline-run-...` con una cuenta gratuita.")
except Exception as e:
    print("W&B omitido (opcional):", repr(e))

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


wandb: Detected [groq] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


Corrida registrada en W&B (offline). Carpeta: c:\Users\stron\Desktop\modulo1IA\M3\wandb
Para verla en el panel web: `wandb sync wandb/offline-run-...` con una cuenta gratuita.


## 21 · Lectura honesta

> ⚠️ **Pendiente de completar.** Esta sección se llena DESPUÉS de correr el notebook completo y
> mirar los números reales de `tabla_comparativa`, `ragas_m3` y (si corrió) `anclaje_juez_m3`.
> No hay lectura honesta sin datos — es exactamente el punto que M2 nos enseñó. Guía de qué
> responder aquí, con la evidencia de arriba:

- [ ] **¿Subieron los aciertos de dominio gold con RAG?** Comparar `aciertos_gold` de
      `0_baseline_sin_rag` vs. los tres sistemas RAG en `tabla_comparativa`.
- [ ] **¿Qué tanto pagó el corpus externo?** Mirar específicamente `aciertos_gold_EXTERNO`
      (los 12 casos `gold-ica-*`, incluidos café y cacao) — ahí es donde el baseline sin RAG
      *no tiene ninguna posibilidad* de acertar por diseño. Si RAG tampoco acierta ahí, es una
      señal de que el retrieval o el prompt de generación (§12) fallan, no de que "RAG no
      sirve".
- [ ] **¿Hybrid+reranking superó al RAG ingenuo?** Comparar `1_rag_ingenuo` vs.
      `2_rag_avanzado` en `tabla_comparativa`, y `context_precision`/`context_recall` en
      `ragas_m3` (§18) — ¿la mejora en aciertos coincide con una mejora en esas dos métricas?
- [ ] **¿Las herramientas de dominio (§13–14) valieron la pena?** Comparar `3_rag_agentico`
      contra `2_rag_avanzado` en `tabla_comparativa`, por partes: (a) **adversariales** — ¿subió
      `abstuvo_adv`? Si sí, recordar que lo produce el guardrail determinista de
      `diagnostico_diferencial`, no el modelo (el mismo modelo que en M2 tuvo 0/3); (b) **gold**
      — ¿bajó `patogeno_conflictivo` en el detalle (dejó de nombrar patógenos de otro cultivo)?
      ¿subió `menciona_patogeno`? Eso sería el efecto de filtrar por cultivo + ficha exacta;
      (c) ¿cuánto de eso aportó el planificador y cuánto las redes de seguridad deterministas?
      Revisar las trazas de los 4 casos demo de §14; y (d) cuánta latencia agregó.
- [ ] **¿Qué pasó con `adv-04` (ambigüedad de retrieval)?** Es la prueba de ruido diseñada a
      propósito: dos documentos (roya de maíz, roya de café) igual de plausibles por
      significado. Revisar en el detalle si el sistema pidió aclarar el cultivo o mezcló los
      dos manejos con seguridad.
- [ ] **¿El anclaje en 3 desapareció con el tercer juez?** Si corrió la §19 (requiere
      `GROQ_API_KEY`), comparar `juez_local_1.5B` vs. `juez_api_70B` en `anclaje_juez_m3` para
      el probe `POBRE/disparate`: ¿el juez de API castiga más el disparate que el local?
- [ ] **¿El fix de `menciona_patogeno` (§5) cambió algo en la práctica** más allá de la prueba
      de regresión sintética? Revisar si algún caso real del detalle de `2_rag_avanzado` se ve
      afectado por la exclusión de tokens del cultivo.
- [ ] **La tensión metodológica del §13/§14** (el agente y el juez comparten el checkpoint
      `Qwen2.5-1.5B-Instruct`): ¿el `juez` del sistema agéntico se ve sospechosamente alto
      comparado con su `puntaje_dominio` (Dimensión 3, que no depende del juez)? Si hay una
      brecha grande, es la auto-preferencia de la que habla §23 (Limitaciones).

*(Completar con 5-8 líneas de lectura honesta citando los números reales, igual que el §8 de M2.)*

## 22 · Reproducibilidad — cómo se corre

1. Clonar el repo. La carpeta `M3/` ya trae: `notebookM3.ipynb` (este), `eval_set.json`
   (heredado, sin tocar) + `eval_set_m3_extra.json` (nuevo), `RUBRICA.md`, el adaptador
   `mi-modelo-lora/` y `datos/` — incluye `corpus_ica_colombia.json` (metadata) y los **50 PDF
   del corpus RAG** en `datos/pdfs/entrenamiento/` (38) y `datos/pdfs/externo/` (12). Si la
   carpeta `pdfs/` no viene completa, el notebook mismo los regenera con `reportlab` (§6).
2. Abrir `notebookM3.ipynb` en Colab (**T4 recomendada** — hay más cómputo que M2: 4 sistemas
   × 27 casos, más RAGAS y el reranker) o Jupyter local con GPU.
3. *(Opcional)* Configurar `GROQ_API_KEY` (Secretos de Colab o variable de entorno) **antes**
   de correr, si quieren la §19.
4. **Un solo comando: *Runtime → Run all*.** Los modelos se descargan del Hub de Hugging Face
   (el mismo `Qwen2.5-0.5B-Instruct` + `mi-modelo-lora`, más `Qwen2.5-1.5B-Instruct` y
   `SmolLM2-1.7B-Instruct` de M2, más el reranker `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1`).
5. Produce `scorecard_rag_m3.csv`, `scorecard_rag_m3.json`, `eval_set_m3.json`,
   `RUBRICA_snapshot_m3.txt`, `ragas_m3.csv` y (si hay clave) `anclaje_juez_m3.csv` en `M3/`.

**Qué garantiza los mismos números entre corridas (heredado de M2):**

- `SEED = 42` en `random`, `numpy`, `torch`, `transformers.set_seed`; `CUBLAS_WORKSPACE_CONFIG`.
- Los 4 sistemas y ambos jueces locales usan decodificación **greedy** (`do_sample=False`) o
  puntaje por valor esperado sobre logits (sin muestreo) → sin varianza entre corridas en la
  misma GPU.
- Versiones de librerías y `revision` (commit) de cada modelo quedan en `scorecard_rag_m3.json`.

**Qué NO es determinista, por diseño:**

- El **tercer juez de API** (§19) depende de un servicio externo (Groq): puede cambiar de
  versión de modelo o tener variación menor entre llamadas aunque `temperature=0`. Por eso se
  reporta aparte (`anclaje_juez_m3.csv`), no se mezcla con el scorecard principal.
- `multi_query` (§11) y el `_juez_si_no` de RAGAS (§18) usan el juez local con `do_sample=False`
  también, así que **sí** son deterministas — pero dependen de la versión exacta del checkpoint
  de `Qwen2.5-1.5B-Instruct` publicada en el Hub (mismo caveat que M2: guardamos el `revision`).

## 23 · Limitaciones de esta evaluación

- **Las fichas PDF de `datos/pdfs/externo/` no son documentos oficiales.** Fueron redactadas
  por el equipo para este ejercicio académico, inspiradas en información pública de ICA,
  AGROSAVIA, Cenicafé y Fedecacao, pero **no citan resoluciones ni cifras exactas** (siguiendo
  la misma política de la base de M1/M2 de no dar dosis exactas) y no deben tratarse como
  norma vigente. Esto se declara en cada ficha (pie de página y `fuente`) y aquí.
- **Los PDF de `datos/pdfs/` son generados por el propio equipo** (no escaneos ni documentos
  encontrados "en la calle"): el texto que extrae `pypdf` es limpio por construcción (una sola
  columna, sin tablas ni imágenes). Un PDF real de una entidad gubernamental suele traer
  columnas, encabezados repetidos o texto en imágenes que degradan la extracción — ese ruido
  de "PDF real" no está representado aquí.
- **La abstención del sistema agéntico es un guardrail determinista, no una capacidad del
  modelo.** Cuando `diagnostico_diferencial` dice que el cultivo no está cubierto o que falta,
  la respuesta es una plantilla fija; esa plantilla contiene, por diseño, las frases que buscan
  los detectores `_ABST_*` de la Dimensión 3. Un `abstuvo_adv` alto en `3_rag_agentico` mide que
  el guardrail se dispara en el caso correcto — no que el modelo de 0.5B haya aprendido a decir
  "no sé". La cobertura se decide por **nombre de cultivo** (`ALIAS_CULTIVO`): un cultivo con un
  nombre regional que no está en la lista se trata como no cubierto.
- **`adv-01` (roya del café) cambió de naturaleza con el corpus de M3.** Su criterio, escrito en
  M2, exige abstención porque el café está fuera de las 38 clases. Pero ahora la biblioteca SÍ
  trae una ficha de Cenicafé, y `diagnostico_diferencial` reporta el café como cubierto (por
  fuente externa): responder citando esa ficha es defendible, y la regla estricta de la
  Dimensión 3 lo puede contar como fallo si la respuesta receta un fungicida. Lo dejamos sin
  cambiar para no mover la vara entre M2 y M3, y lo leemos caso por caso en §21.
- **El agente (§13–14) y el juez principal comparten el mismo checkpoint**
  (`Qwen2.5-1.5B-Instruct`) como "cerebro" de razonamiento. Esto es una tensión metodológica
  real: si el juez tiende a valorar más las trazas de razonamiento que él mismo generaría, el
  `llm_juez_prom` de `3_rag_agentico` podría estar inflado por auto-preferencia (el mismo
  fenómeno que M2 midió entre sistema y juez, pero aquí entre razonador-y-evaluador). Por eso
  el **acierto de dominio (Dimensión 3)**, que no depende del juez, es el criterio que debe
  pesar más al comparar el sistema agéntico contra los otros tres.
- **RAGAS es casero, no la librería oficial.** Usamos el juez local para las sub-preguntas de
  sí/no en vez de `ragas` + un LLM potente vía API; es la misma limitación que reconoce
  explícitamente el lab de S10. Los números son comparables **entre nuestros propios sistemas**
  (mismo método para ambos), no necesariamente contra publicaciones que usan la librería.
- **El tercer juez de API es opcional y depende de un servicio externo.** Si no hay
  `GROQ_API_KEY`, la sección 19 se omite entera y esta entrega queda sin esa evidencia — un
  lector que reproduzca sin clave no verá `anclaje_juez_m3.csv`.
- **27 casos siguen siendo pocos para un intervalo de confianza**, aunque son el doble que en
  M2. Los 12 casos nuevos de fuente externa, en particular, son un *n* de apenas 12 (10
  Colombia + 2 fuera de dominio); un solo caso todavía mueve varios puntos porcentuales el
  promedio de ese subgrupo.
- **El corpus RAG mezcla fuente de entrenamiento y fuente externa en el mismo índice.** Esto es
  intencional (así se comporta un RAG real), pero significa que el retrieval puede traer un
  chunk de *cualquiera* de las dos fuentes para una pregunta de la otra; no forzamos que
  `gold-ica-*` solo recupere de `corpus_ica_colombia.json`.
- **`adv-04` y el resto de los adversariales se puntúan con un detector de patrones (regex),
  igual que en M2** — hereda la misma fragilidad: una respuesta correcta con una redacción que
  el equipo no anticipó puede no disparar ninguna señal de `_ABST_*` y contar como fallo.
- **Dominio de las fuentes.** Las 38 clases originales siguen siendo de extensión agrícola de
  EE. UU.; el corpus externo de Colombia cubre 10 de esas 38 enfermedades más 2 cultivos
  nuevos (café, cacao) — no las 38 clases completas.

*SI4006 · Universidad EAFIT · Módulo 3 — RAG agéntico y evaluación con RAGAS · Entrega M3.*